# Path C+ Option C **9-NODE** — Extended-DAG retrain Stage 1 (+ Stage 2) from-scratch (3 seeds)

> **9-node variant** of `st_cdgm_path_c_option_c.ipynb`. Adds the humid chain
> `Q850`, `W500`, `IVT` (Held-Soden wet closure) to the 6-node dry DAG, for a
> 9-variable causal graph. Four surgical changes vs the 6-node base :
> 1. `HeteroGraphBuilder(..., extended_9node=True)` (Cell 3) ;
> 2. 3 spatial metapaths `Q850/W500/IVT` injected into `CONFIG.encoder.metapaths` (Cell 3) ;
> 3. `convert_sample_to_batch` routes per-node channels + derives an IVT proxy (Cell 3) ;
> 4. `G_phys` built from `VAR_LABELS_9NODE` / `EXPECTED_EDGES_9NODE` (Cell 5).
>
> **Run mode (current)** : `STAGE1_ONLY = False` → **full Stage 1 + Stage 2**,
> **single seed = 42** (`SEEDS = [42]`, the project's canonical seed). NOT
> `finetune_bundle_b` (Stage-1-only / warm-start) — this uses `train_epoch_stage1`
> + `train_epoch_stage2_cached` from-scratch. Output dir `oracle_9node/`
> (does NOT clobber the 6-node `oracle_full/`).
>
> **Persistence + monitoring** : every epoch atomically writes `epoch_last.pth`
> (tmp+fsync+rename, resume-safe) and appends a structured line to
> `oracle_9node/seed_42/training_log.jsonl`; the best Stage-1 checkpoint is kept
> as `epoch_best_stage1.pth`. A per-epoch health banner prints loss / A_dag norm
> / val MSE / GPU peak / epoch time and **flags anomalies** (NaN loss, A_dag
> collapse, loss or val-MSE rising 3× in a row, epoch slowdown) so a bad run can
> be killed early. `STAGE1_ONLY = True` (+ Cell 6b) remains available for a cheap
> deterministic Stage-1 gate if ever needed.

**Goal** : produce a thesis-grade Path C+ Oracle retrain matching CorrDiff Normal V2 config (`training_config_corrdiff_normal.yaml`, Stage 1 = 15 epochs, Stage 2 = 200 epochs), with Path C+ hyperparameter overrides, K9 temporal split, 3 seeds, and full H1-H5 evaluation pipeline mirrored from `st_cdgm_noncausal_training.ipynb` (see `path_c_plus/audit/NONCAUSAL_EVAL_CONTRACT.md`).

**Code path** : `src.st_cdgm.training.training_loop.train_epoch_stage1` + `train_epoch_stage2` (NOT `finetune_bundle_b` like A1).

**Hyperparameter strategy** : structural CorrDiff config (epochs / batch / scheduler) + Path C+ overrides (lambda_dag_prior=0.40, lambda_l1=0.04->0.005, g_phys_alpha=0.25, dag_grad_gate auto-scale ramp). See `PATHCPLUS_HYPERPARAM_OVERRIDES` in `path_c_plus.scripts.option_c_helpers`.

**Output structure** :
```
/content/drive/MyDrive/climate_data/oracle_full/
  seed_42/
    epoch_last.pth                        (resume)
    final_validation_metrics.json         (cell 61 mirror)
    domain_metrics.json                   (cell 62 mirror)
    eval_samples.npz                      (cell 63 mirror)
    aligned_metrics_ACCESS-CM2_causal.json (cell 64+65, in-dist)
    aligned_metrics_EC-Earth3_causal.json   (cell 64+65, OOD)
    aligned_metrics_NorESM2-MM_causal.json  (cell 64+65, OOD)
    results.json                          (Path C+ specific: Q_phys + verdict)
  seed_7/  ...
  seed_123/ ...
  oracle_full_aggregate.json              (cross-seed H1-H5 verdict + Holm-Bonferroni)
```

**Baseline for H2-H5** : `/content/drive/MyDrive/climate_data/ckpt_noncausal/` (epoch=200, verified COMPLETE)

**Resume semantics** : each seed's training writes `epoch_last.pth` per epoch atomically. If `oracle_full/seed_<n>/results.json` exists, the seed is skipped. To re-run a seed, delete the directory.

**Estimated compute** (refined post-council DS #2 audit) : 28-40h/seed on A100 (Stage 1 ~3-5h, Stage 2 ~25-35h with BS32b cache + EMA), 3 seeds = **~85-120h sequential within PC14 #8 cap**. Eval pipeline ~3-4.5h total. Use Colab Pro+ A100, 1-2 sessions per seed.

If wall-time exceeds 120h, PC14 amendment "Option C-bis" is required (consult PRE_REGISTRATION.md).

In [ ]:
# >>> Cell 1 : Bootstrap Colab + git sync
import os, sys, subprocess, time, shlex
from pathlib import Path

GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "four-node-causal"
LOCAL_PROJECT = "/content/climate_data"
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    print(f"$ {cmd}"); t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0; print(f"  rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc

if _IS_COLAB:
    _T0 = time.time()
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")
    if GIT_PULL_ON_RESUME:
        _run(f"git -C {LOCAL_PROJECT} fetch --depth=200 origin {GIT_BRANCH}", timeout=180)
        try:
            cur = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --abbrev-ref HEAD")).decode().strip()
        except Exception:
            cur = ""
        if cur != GIT_BRANCH:
            _run(f"git -C {LOCAL_PROJECT} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}", timeout=30)
        else:
            _run(f"git -C {LOCAL_PROJECT} reset --hard origin/{GIT_BRANCH}", timeout=30)
        head_sha = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --short HEAD")).decode().strip()
        head_msg = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} log -1 --pretty=%s")).decode().strip()
        print(f"   HEAD = {head_sha}  ({head_msg})")
    os.chdir(project_path)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            print("OK Imports critiques — pip install sauté.")
        except ImportError as e:
            print(f"pip install requis : {e}")
            EXTRA_DEPS = ["omegaconf==2.3.0", "hydra-core==1.3.2", "diffusers==0.36.0",
                          "transformers==4.57.6", "accelerate==1.12.0", "huggingface-hub==0.36.0",
                          "safetensors==0.7.0", "xbatcher", "webdataset", "cftime", "h5netcdf",
                          "numcodecs", "torch-geometric", "xformers"]
            _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
                 + " ".join(shlex.quote(p) for p in EXTRA_DEPS), timeout=600)
            _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
                 f"--no-deps -e {LOCAL_PROJECT}", timeout=120)
    print(f"\nBootstrap terminé en {time.time() - _T0:.1f}s.")
else:
    _here = Path.cwd()
    for _c in [_here, *_here.parents]:
        if (_c / "config" / "training_config.yaml").exists() and (_c / "setup.py").exists():
            if _c != _here:
                os.chdir(_c)
            break
    print("Hors Colab")


In [ ]:
# >>> Cell 2 : Config CorrDiff Normal V2 + Path C+ hyperparam strategy
#
# AI Eng CRITICAL BUG fix (council pre-review GO-WITH-CHANGE) :
# Previous version wrote `lambda_l1_start / lambda_l1_end / dag_gate_warmup_*`
# onto `ts_cfg.stage1` -- but `train_epoch_stage1` (training_loop.py:1702-1739)
# reads only SCALAR `lambda_l1` + `dag_grad_gate_value`. The cosine annealing
# + gate ramp live in `schedule_lambdas` (finetune_stage1_bundle_b.py:173-231),
# which Option C training path does NOT call. Those keys were DEAD CODE.
#
# Fix : (a) don't write the schedule keys onto ts_cfg here; (b) cell 6 will
# import `schedule_lambdas` + `DEFAULT_HYPERPARAMS` and call it per-epoch to
# produce the annealed lambda_l1 + dag_grad_gate values that are passed into
# `train_epoch_stage1` as scalars per epoch.
#
# This cell just sets the SCALAR Path C+ overrides that `train_epoch_stage1`
# does read directly : lambda_dag_prior + g_phys_alpha. The lambda_l1 + gate
# are managed per-epoch in cell 6 via schedule_lambdas.
#
from omegaconf import OmegaConf
from path_c_plus.scripts.option_c_helpers import (
    PATHCPLUS_HYPERPARAM_OVERRIDES,
    PC13_NEW_PARAMS_ALLOWLIST,
)
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)
if GPU_PROFILE.get("profile_id") not in ("a100", "h100"):
    print(f"\nWARN Option C was budgeted for A100. Detected: {GPU_PROFILE.get('profile_id')}")
    print("     T4/V100 will multiply wall-time by 5-8x.")

# Load CorrDiff Normal V2 config (causal variant)
CONFIG = OmegaConf.load("config/training_config.yaml")
_corrdiff = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

# === 9-NODE variant flags ===========================================
# EXTENDED_9NODE : turns on the humid chain (Q850/W500/IVT) end-to-end
#   (builder + metapaths + node routing + G_phys). All 9-node wiring is
#   gated behind this single flag so the file degrades gracefully to the
#   exact 6-node behaviour if set False.
# STAGE1_ONLY    : FULL run (Stage 1 + Stage 2). The user dropped the
#   Stage-1-only gate -> we commit the full two-stage on a SINGLE best seed.
#   (Left as a switch in case a cheap Stage-1 probe is ever wanted again.)
EXTENDED_9NODE = True
STAGE1_ONLY = False   # <-- FULL two-stage (Stage 1 + Stage 2), single seed
print(f"\n[9NODE] EXTENDED_9NODE={EXTENDED_9NODE}  STAGE1_ONLY={STAGE1_ONLY} "
      f"(False = full Stage 1 + Stage 2)")

# Apply GPU profile (batch_size, amp, num_workers)
CONFIG.training.batch_size = GPU_PROFILE["batch_size"]
CONFIG.training.use_amp = GPU_PROFILE["use_amp"]
CONFIG.training.num_workers = GPU_PROFILE["num_workers"]

# === Path C+ scalar hyperparameter override (read directly by train_epoch_stage1) ===
# These are the values that A1 used and produced Q_phys_cont = 0.521.
# Schedule values (lambda_l1, dag_grad_gate) are NOT set here -- they're
# computed per-epoch by schedule_lambdas() in cell 6.
print("\n=== Path C+ scalar hyperparameter override (vs CorrDiff Normal V2 defaults) ===")
ts_cfg = CONFIG.two_stage
_SCALAR_OVERRIDES = {
    "lambda_dag_prior": PATHCPLUS_HYPERPARAM_OVERRIDES["lambda_dag_prior"],   # 0.40 (was 0.05)
    "g_phys_alpha": PATHCPLUS_HYPERPARAM_OVERRIDES["g_phys_alpha"],           # 0.25 (was 0.20)
}
for key, new_val in _SCALAR_OVERRIDES.items():
    old_val = ts_cfg.stage1.get(key, "<MISSING>")
    ts_cfg.stage1[key] = new_val
    print(f"  stage1.{key:25s} : {old_val} -> {new_val}")

print(f"\n  NB schedule_lambdas() will produce per-epoch :")
print(f"  - lambda_l1 (cosine 0.04 -> 0.005)")
print(f"  - dag_grad_gate (auto-scale ramp 0 -> 1 over epochs 5-20)")
print(f"  See cell 6 for the per-epoch schedule_lambdas() wiring.")

# Confirm run_variant = causal
assert ts_cfg.get("run_variant") == "causal", (
    f"run_variant must be 'causal' for Option C, got {ts_cfg.get('run_variant')!r}"
)
print(f"\n  run_variant = {ts_cfg.run_variant}")

# J29 eval/training safety : corrdiff_normal.yaml sets cfg_scale=1.5 (V4 legacy) but
# our causal stack uses scheduler_type=edm_karras where CFG is NOT implemented in
# _sample_edm_karras. Force cfg_scale=1.0 here so Cell 8 eval never passes 1.5.
_sch = str(CONFIG.diffusion.get("scheduler_type", "edm_karras"))
_cfg_old = float(CONFIG.diffusion.get("cfg_scale", 1.0))
if _sch == "edm_karras" and _cfg_old > 1.0 + 1e-9:
    CONFIG.diffusion.cfg_scale = 1.0
    print(f"  [J29] diffusion.cfg_scale : {_cfg_old} -> 1.0 (edm_karras has no CFG)")
print(f"  stage1.epochs_max = {ts_cfg.stage1.get('epochs_max')}")
print(f"  stage2.epochs_max = {ts_cfg.stage2.get('epochs_max')}")
print(f"  data.stride = {CONFIG.data.stride}")
print(f"  training.batch_size = {CONFIG.training.batch_size}")

# ORACLE_FULL_DIR (Path C+ Option C output) -- 9-node writes to oracle_9node/
# so the 6-node oracle_full/ artifacts are never clobbered.
_ORACLE_SUBDIR = "oracle_9node" if EXTENDED_9NODE else "oracle_full"
ORACLE_FULL_DIR = Path(f"/content/drive/MyDrive/climate_data/{_ORACLE_SUBDIR}")
ORACLE_FULL_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n[Option C] ORACLE_FULL_DIR = {ORACLE_FULL_DIR}")
print(f"[Option C] Existing seed dirs : {sorted(p.name for p in ORACLE_FULL_DIR.iterdir() if p.is_dir())}")

# Noncausal baseline (for H2-H5)
CKPT_NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")
assert CKPT_NONCAUSAL_DIR.exists(), f"Noncausal baseline missing : {CKPT_NONCAUSAL_DIR}"
print(f"[Option C] CKPT_NONCAUSAL_DIR = {CKPT_NONCAUSAL_DIR}  (H2-H5 baseline)")


In [ ]:
# >>> Cell 3 : Pipeline + K9 temporal split + builder + datasets + iterate_batches
#
# Mirrors noncausal_training cells 26-29 + 33 + 41, with two structural changes:
#   1. K9 temporal split (1980-2009 train / 2010-2011 val / 2012-2013 test /
#      2014 holdout) passed as YYYY-MM-DD strings to NetCDFDataPipeline.
#   2. iterate_batches + convert_sample_to_batch ported verbatim from cell 41
#      (noncausal line 30403-30440) so it takes (builder, device) explicitly
#      with no hidden globals.
#
# IMPORTANT : do NOT shuffle val_dataloader, do NOT change BATCH_SIZE between
# seeds -- training loop has BS32b cache keyed on samples per epoch.

import os
import sys
import torch
import xarray as xr
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader as _DataLoader, IterableDataset

from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder

# -----------------------------------------------------------------------------
# 1. Data paths (Drive-resident on Colab) -- mirrors noncausal cell 17/18 layout
#    /content/drive/MyDrive/climate_data/data/
#      train/predictor_ACCESS-CM2_hist.nc
#      train/pr_ACCESS-CM2_hist.nc
#      test/EC-Earth3_histupdated_compressed.nc
#      test/EC-Earth3_historical_precip_compressed.nc
#      test/NorESM2-MM_histupdated_compressed.nc
#      test/NorESM2-MM_historical_precip_compressed.nc
#      static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc
#      normalization_coefs/mean_1974_2011.nc
#      normalization_coefs/std_1974_2011.nc
#
# Includes : Zenodo download fallback if missing + BS32 SSD copy for fast I/O.
# -----------------------------------------------------------------------------
import shutil as _shutil
import urllib.request as _ureq
import urllib.error as _uerr

_ON_COLAB = "google.colab" in sys.modules or Path("/content").exists()
DATA_ROOT_DRIVE = Path("/content/drive/MyDrive/climate_data/data")
DATA_ROOT_LOCAL = Path.cwd() / "data" / "raw"

if _ON_COLAB and DATA_ROOT_DRIVE.parent.parent.exists():
    DATA_ROOT = DATA_ROOT_DRIVE
    print(f"[Cell 3] DATA_ROOT = Drive ({DATA_ROOT})")
else:
    DATA_ROOT = DATA_ROOT_LOCAL
    print(f"[Cell 3] DATA_ROOT = local ({DATA_ROOT.resolve()})")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Ensure train/ static_predictors/ normalization_coefs/ exist
(DATA_ROOT / "train").mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "static_predictors").mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "normalization_coefs").mkdir(parents=True, exist_ok=True)

# Zenodo URLs for the 2 training files (the test/OOD files are NOT on Zenodo;
# they must already be on Drive from the noncausal run).
URL_ZENODO_HR = "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1"
URL_ZENODO_LR = "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1"

def _stream_dl(url, dest, retries=5, chunk=1024*1024):
    """Streaming download with .part + os.replace + resume + backoff."""
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    import time as _t
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = _ureq.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
        try:
            with _ureq.urlopen(req, timeout=30) as resp:
                total = resp.length or (int(resp.headers["Content-Length"])
                                         if resp.headers.get("Content-Length") else None)
                mode = "ab" if already > 0 else "wb"
                with open(part, mode) as f:
                    downloaded = already
                    last_log = _t.time(); last_bytes = downloaded
                    while True:
                        c = resp.read(chunk)
                        if not c: break
                        f.write(c); downloaded += len(c)
                        now = _t.time()
                        if now - last_log >= 5:
                            speed = (downloaded - last_bytes) / (now - last_log) / 1e6
                            print(f"    {downloaded/1e6:7.1f} MB -- {speed:5.1f} MB/s")
                            last_log = now; last_bytes = downloaded
            os.replace(part, dest)
            print(f"  OK {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return True
        except (_uerr.HTTPError, _uerr.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2 ** attempt)
            print(f"  WARN {type(e).__name__}: {e} -- retry in {wait}s")
            import time; time.sleep(wait)
    return False

# Ensure the 2 training files exist on Drive (download from Zenodo if missing)
_lr_drive = DATA_ROOT / "train" / "predictor_ACCESS-CM2_hist.nc"
_hr_drive = DATA_ROOT / "train" / "pr_ACCESS-CM2_hist.nc"
if not _lr_drive.exists():
    print(f"[Cell 3] downloading LR from Zenodo -> {_lr_drive} ...")
    _stream_dl(URL_ZENODO_LR, _lr_drive)
if not _hr_drive.exists():
    print(f"[Cell 3] downloading HR from Zenodo -> {_hr_drive} ...")
    _stream_dl(URL_ZENODO_HR, _hr_drive)

# BS32 SSD copy : Drive NetCDF mmap latency dominates training time. Copy the
# training .nc files to local SSD once per session. Test files stay on Drive.
_DATA_ROOT_LOCAL_SSD = Path("/content/data_local")
_BS32_ENABLED = bool(globals().get("DATA_LOCAL_SSD", True))
if (_ON_COLAB and _BS32_ENABLED and DATA_ROOT == DATA_ROOT_DRIVE):
    print(f"[Cell 3] BS32 : copying training data to SSD {_DATA_ROOT_LOCAL_SSD}...")
    _to_copy = [
        ("train/predictor_ACCESS-CM2_hist.nc",   "train/predictor_ACCESS-CM2_hist.nc"),
        ("train/pr_ACCESS-CM2_hist.nc",          "train/pr_ACCESS-CM2_hist.nc"),
        ("static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc",
         "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc"),
        ("normalization_coefs/mean_1974_2011.nc", "normalization_coefs/mean_1974_2011.nc"),
        ("normalization_coefs/std_1974_2011.nc",  "normalization_coefs/std_1974_2011.nc"),
    ]
    import time as _tt
    _t0 = _tt.time(); _bytes = 0
    for _rel_src, _rel_dst in _to_copy:
        _src = DATA_ROOT_DRIVE / _rel_src
        _dst = _DATA_ROOT_LOCAL_SSD / _rel_dst
        _dst.parent.mkdir(parents=True, exist_ok=True)
        if not _src.exists():
            print(f"   WARN missing on Drive (skip): {_src}")
            continue
        if _dst.exists() and _dst.stat().st_size == _src.stat().st_size:
            print(f"   already on SSD: {_dst.name} ({_dst.stat().st_size/1e6:.0f} MB)")
            continue
        print(f"   copying {_src.name}...", flush=True)
        _shutil.copy2(_src, _dst)
        _bytes += _dst.stat().st_size
    print(f"[Cell 3] BS32 done : {_bytes/1e9:.2f} GB in {_tt.time()-_t0:.1f}s -> DATA_ROOT = SSD")
    DATA_ROOT = _DATA_ROOT_LOCAL_SSD

# Final paths (noncausal naming convention)
LR_PATH = str(DATA_ROOT / "train" / "predictor_ACCESS-CM2_hist.nc")
HR_PATH = str(DATA_ROOT / "train" / "pr_ACCESS-CM2_hist.nc")
_static_p = DATA_ROOT / "static_predictors" / "ERA5_eval_ccam_12km.198110_NZ_Invariant.nc"
STATIC_PATH = str(_static_p) if _static_p.exists() else None
_mean_p = DATA_ROOT / "normalization_coefs" / "mean_1974_2011.nc"
_std_p  = DATA_ROOT / "normalization_coefs" / "std_1974_2011.nc"
MEAN_PATH = str(_mean_p) if _mean_p.exists() else None
STD_PATH  = str(_std_p)  if _std_p.exists()  else None

# Final-check : training files must exist (we can't proceed without them)
assert Path(LR_PATH).exists(), f"LR file missing : {LR_PATH}"
assert Path(HR_PATH).exists(), f"HR file missing : {HR_PATH}"

print(f"[Cell 3] LR     = {LR_PATH}")
print(f"[Cell 3] HR     = {HR_PATH}")
print(f"[Cell 3] STATIC = {STATIC_PATH}")
print(f"[Cell 3] MEAN   = {MEAN_PATH}")
print(f"[Cell 3] STD    = {STD_PATH}")

# -----------------------------------------------------------------------------
# 2. K9 temporal split (PRE_REGISTRATION.md PC9 -- LOCKED at A1 commit 78a3783)
# -----------------------------------------------------------------------------
K9_DATES = {
    "train":   ["1980-01-01", "2009-12-31"],  # 30 years
    "val":     ["2010-01-01", "2011-12-31"],  # 2 years
    "test":    ["2012-01-01", "2013-12-31"],  # 2 years
    "holdout": ["2014-01-01", "2014-12-31"],  # 1 year
}
print(f"\n[Cell 3] K9 temporal split (locked at A1) :")
for _split, _bounds in K9_DATES.items():
    print(f"         {_split:8s} = {_bounds[0]} -> {_bounds[1]}")

# -----------------------------------------------------------------------------
# 3. Pipeline (matches noncausal cell 27 args, plus K9 dates)
# -----------------------------------------------------------------------------
SEQ_LEN = int(CONFIG.data.seq_len)
BASELINE_STRATEGY = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR = int(CONFIG.data.baseline_factor)
NORMALIZE = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY = str(CONFIG.data.nan_fill_strategy)

_default_lr = ["q_500", "q_850", "u_500", "u_850", "v_500", "v_850", "t_500", "t_850"]
_default_hr = ["pr"]
LR_VARIABLES = list(CONFIG.data.lr_variables) if CONFIG.data.get("lr_variables") else _default_lr
HR_VARIABLES = list(CONFIG.data.hr_variables) if CONFIG.data.get("hr_variables") else _default_hr
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get("static_variables") else (
    ["orog", "he", "vegt"] if STATIC_PATH else None
)

# Validate variables actually present in the file (cell 27 logic)
_lr_avail = set(xr.open_dataset(LR_PATH).data_vars)
_hr_avail = set(xr.open_dataset(HR_PATH).data_vars)
if not set(LR_VARIABLES).issubset(_lr_avail):
    LR_VARIABLES = [v for v in LR_VARIABLES if v in _lr_avail] or sorted(_lr_avail)[:8]
if not set(HR_VARIABLES).issubset(_hr_avail):
    HR_VARIABLES = [sorted(_hr_avail)[0]]

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH,
    hr_path=HR_PATH,
    static_path=STATIC_PATH,
    seq_len=SEQ_LEN,
    baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR,
    normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES,
    hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH if (MEAN_PATH and os.path.exists(MEAN_PATH)) else None,
    stds_path=STD_PATH if (STD_PATH and os.path.exists(STD_PATH)) else None,
    train_start_date=K9_DATES["train"][0], train_end_date=K9_DATES["train"][1],
    val_start_date=K9_DATES["val"][0], val_end_date=K9_DATES["val"][1],
    test_start_date=K9_DATES["test"][0], test_end_date=K9_DATES["test"][1],
    temporal_holdout_start_date=K9_DATES["holdout"][0],
    temporal_holdout_end_date=K9_DATES["holdout"][1],
)
print(f"\n[Cell 3] NetCDFDataPipeline ready -- LR={pipeline.lr_dataset.dims}, "
      f"HR={pipeline.hr_dataset.dims}")

# -----------------------------------------------------------------------------
# 4. Train + Val datasets (K9 splits, stride=2 from CorrDiff Normal V2)
# -----------------------------------------------------------------------------
train_dataset = pipeline.build_sequence_dataset(
    split="train", seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True,
)
val_dataset = pipeline.build_sequence_dataset(
    split="val", seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True,
)
dataset = train_dataset  # Backward-compat alias for cells that probe `dataset`

# Sample probe to expose channel counts to downstream cells
_sample_probe = next(iter(train_dataset))
print(f"\n[Cell 3] Sample shapes : lr={tuple(_sample_probe['lr'].shape)}, "
      f"residual={tuple(_sample_probe['residual'].shape)}")
sample = _sample_probe  # cell 4 (stack constructors) reads this

# -----------------------------------------------------------------------------
# 5. DataLoaders (BS32b expects DataLoader yielding lists when collate=lambda x:x)
# -----------------------------------------------------------------------------
BATCH_SIZE = int(CONFIG.training.batch_size)  # already overridden by GPU profile
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY = bool(torch.cuda.is_available())

_loader_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, collate_fn=lambda x: x,
)
if NUM_WORKERS > 0:
    _loader_kwargs["persistent_workers"] = True
    _loader_kwargs["prefetch_factor"] = 2

train_dataloader = _DataLoader(
    train_dataset,
    shuffle=(not isinstance(train_dataset, IterableDataset)),
    **_loader_kwargs,
)
val_dataloader = _DataLoader(val_dataset, shuffle=False, **_loader_kwargs)
print(f"[Cell 3] DataLoaders : BS={BATCH_SIZE}, NW={NUM_WORKERS}, pin={PIN_MEMORY}")

# -----------------------------------------------------------------------------
# 6. Graph builder (noncausal cell 33 verbatim)
# -----------------------------------------------------------------------------
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)

# --- 9-node : inject the 3 humid-chain spatial metapaths into the config -----
# _build_encoder (cell 4) iterates CONFIG.encoder.metapaths and keeps those whose
# src/tgt survive the graph node filter. The builder (extended_9node=True) adds
# Q850/W500/IVT as dynamic node types with (nt, "spat_adj", nt) self edges, so
# these 3 metapaths produce 3 extra intelligible variables -> num_vars 6 -> 9.
if EXTENDED_9NODE:
    from omegaconf import OmegaConf as _OC9
    _existing_mp = {m.name for m in CONFIG.encoder.metapaths}
    _humid_mp = [
        {"name": "Q850_spat", "src": "Q850", "relation": "spat_adj", "target": "Q850", "pool": "mean"},
        {"name": "W500_spat", "src": "W500", "relation": "spat_adj", "target": "W500", "pool": "mean"},
        {"name": "IVT_spat",  "src": "IVT",  "relation": "spat_adj", "target": "IVT",  "pool": "mean"},
    ]
    OmegaConf.set_struct(CONFIG, False)
    for _m in _humid_mp:
        if _m["name"] not in _existing_mp:
            CONFIG.encoder.metapaths.append(_OC9.create(_m))
    print(f"[Cell 3] 9-node metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}")

builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f"[Cell 3] HeteroGraphBuilder ready -- dyn={builder.dynamic_node_types}, "
      f"static={builder.static_node_types}")

# -----------------------------------------------------------------------------
# 7. iterate_batches + convert_sample_to_batch (verbatim port of noncausal cell 41)
# -----------------------------------------------------------------------------
# --- 9-node : per-node channel routing + IVT proxy --------------------------
# The RCN drivers (lr_tensor) ALWAYS keep the full LR channel set so
# RCN_DRIVER_DIM / reconstruction_dim are unchanged vs 6-node. Only the
# ENCODER node features are routed : the humid nodes receive physically
# meaningful sub-channels, and IVT is derived (Held-Soden moisture flux proxy).
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ("q_850", "q_500", "q_250") if v in _VI]
_W_IDX = [_VI[v] for v in ("w_850", "w_500", "w_250") if v in _VI]
_IVT_LEVELS = [lev for lev in ("850", "500", "250")
               if f"q_{lev}" in _VI and f"u_{lev}" in _VI and f"v_{lev}" in _VI]
print(f"[Cell 3] 9-node routing : Q_IDX={_Q_IDX}, W_IDX={_W_IDX}, IVT_levels={_IVT_LEVELS}")


def _compute_ivt_nodes(lr0):
    """IVT proxy per LR node from already-normalized channels.

    Held-Soden : vertically-integrated moisture flux ~ sum_l q_l * |wind_l|.
    NOTE (approximation, cf. PHASE0_SPEC_9NODE.md S2) : computed in z-scored
    space (not raw kg/kg * m/s), then re-standardized to a comparable scale.
    This is a *proxy of a proxy* good enough for the Stage-1 capacity gate;
    a physically exact IVT (raw fields + own normalization) is the Stage-2
    upgrade if the gate passes.
    """
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, _VI[f"q_{lev}"]]
        u = lr0[:, _VI[f"u_{lev}"]]
        v = lr0[:, _VI[f"v_{lev}"]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = torch.zeros(lr0.shape[0], device=lr0.device, dtype=lr0.dtype)
    acc = (acc - acc.mean()) / (acc.std() + 1e-6)
    return acc.unsqueeze(1)  # [N, 1]


def convert_sample_to_batch(sample, builder, device):
    """Convert one dataloader sample -> dict expected by train_epoch_stage{1,2}."""
    lr_seq = sample["lr"]  # [seq_len, channels, lat, lon]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)   # drivers : full LR (unchanged)
    lr0 = lr_nodes_steps[0]
    # 6-node default : every dynamic node gets the full LR (identical to base).
    # 9-node : humid nodes get routed sub-channels / derived IVT.
    if EXTENDED_9NODE:
        _ivt_nodes = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == "Q850":
                dynamic_features[nt] = lr0[:, _Q_IDX] if _Q_IDX else lr0
            elif nt == "W500":
                dynamic_features[nt] = lr0[:, _W_IDX] if _W_IDX else lr0
            elif nt == "IVT":
                dynamic_features[nt] = _ivt_nodes
            else:  # GP850 / GP500 / GP250 -> full LR (6-node behaviour preserved)
                dynamic_features[nt] = lr0
    else:
        dynamic_features = {nt: lr0 for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        "lr": lr_tensor,
        "residual": sample["residual"],
        "baseline": sample.get("baseline"),
        "hetero": hetero,
        "time": sample.get("time"),
    }


def iterate_batches(dataloader, builder, device):
    """Iter that converts each dataloader sample (or list) into a batch dict."""
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        converted = [convert_sample_to_batch(s, builder, device) for s in batch_list]
        yield converted


# Configuration vars consumed by cell 4 (stack) + cell 6 (training)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else CONFIG.training.device)
RCN_DRIVER_DIM = sample["lr"].shape[1]
hr_channels = sample["residual"].shape[1]
HIDDEN_DIM = int(CONFIG.encoder.hidden_dim)
CONDITIONING_DIM = int(CONFIG.encoder.conditioning_dim)
DIFFUSION_STEPS = int(CONFIG.diffusion.steps)

print(f"\n[Cell 3] DEVICE={DEVICE}, RCN_DRIVER_DIM={RCN_DRIVER_DIM}, hr_channels={hr_channels}")
print(f"[Cell 3] HIDDEN_DIM={HIDDEN_DIM}, CONDITIONING_DIM={CONDITIONING_DIM}, "
      f"DIFFUSION_STEPS={DIFFUSION_STEPS}")


In [ ]:
# >>> Cell 4 : Stack constructors (encoder + RCN + regression_head + diffusion)
#
# Defines `build_fresh_stack(seed)` -> per-seed factory called from cell 6.
# Each call resets torch + numpy RNG state, then instantiates the full causal
# stack from CONFIG (CorrDiff Normal V2 + Path C+ overrides). No warm-start.
#
# Mirrors noncausal_training cells 36 (encoder/RCN), 37 (GraphToGridDecoder for
# causal variant), 38 (diffusion in causal_concat mode), with the addition of
# `Sprint 1/2` projector + (optional) HR identifiability head per the YAML.

from omegaconf import OmegaConf as _OC
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder,
    IntelligibleVariableConfig,
    SpatialConditioningProjector,
    CausalConditioningProjector,
    HRTargetIdentifiabilityHead,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.models.regression_head import GraphToGridDecoder


def _build_encoder(builder, CONFIG, DEVICE):
    """Encoder constructed from metapaths present in the graph (cell 36 logic)."""
    allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
    encoder_configs = []
    for _mp in CONFIG.encoder.metapaths:
        _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
        if _src in allowed_nodes and _tgt in allowed_nodes:
            encoder_configs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_src, _rel, _tgt),
                pool=_mp.get("pool", "mean"),
            ))
    if not encoder_configs:
        raise RuntimeError(
            "No metapath survives the graph filter -- check include_mid_layer + "
            "metapath names in CONFIG.encoder"
        )
    if pipeline.get_static_dataset() is not None:
        encoder_configs.append(IntelligibleVariableConfig(
            name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
        ))
    encoder = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(DEVICE)
    return encoder, len(encoder_configs)


def build_fresh_stack(seed: int):
    """Per-seed stack factory. From-scratch, no warm-start.

    Returns dict with keys :
      encoder, rcn_cell, rcn_runner, regression_head, diffusion,
      spatial_projector, hr_ident_head, num_vars, load_audit
    `load_audit = {}` since from-scratch (PC4/PC13 trivially pass).
    """
    print(f"\n[Cell 4] build_fresh_stack(seed={seed}) ...")
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 1. Encoder
    encoder, num_vars = _build_encoder(builder, CONFIG, DEVICE)
    print(f"   encoder : {num_vars} intelligible variables")

    # 2. Causal RCN (causal variant only -- run_variant=causal asserted in cell 2)
    rcn_cell = RCNCell(
        num_vars=num_vars,
        hidden_dim=int(CONFIG.rcn.hidden_dim),
        driver_dim=RCN_DRIVER_DIM,
        reconstruction_dim=RCN_DRIVER_DIM,
        dropout=float(CONFIG.rcn.dropout),
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))
    print(f"   rcn_cell : {sum(p.numel() for p in rcn_cell.parameters()):,} params")

    # 3. Regression head (causal -> GraphToGridDecoder)
    rh_cfg = CONFIG.two_stage.regression_head
    regression_head = GraphToGridDecoder(
        d_model=int(rh_cfg.d_model),
        hr_h=int(CONFIG.diffusion.height),
        hr_w=int(CONFIG.diffusion.width),
        intermediate_h=int(rh_cfg.intermediate_h),
        intermediate_w=int(rh_cfg.intermediate_w),
        n_heads=int(rh_cfg.n_heads),
        refine_channels=int(rh_cfg.refine_channels),
        output_channels=1,
    ).to(DEVICE)
    print(f"   regression_head : {regression_head.num_params():,} params")

    # 4. Diffusion decoder (causal_concat = True for two_stage causal)
    UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ("down_block_types", "up_block_types"):
        if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
            UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
    UNET_KWARGS["projection_class_embeddings_input_dim"] = (
        num_vars * int(CONFIG.diffusion.conditioning_dim)
    )

    _scheduler_type = CONFIG.diffusion.get("scheduler_type", "edm_karras")
    _edm_config = None
    if _scheduler_type == "edm_karras":
        from st_cdgm.models.edm_preconditioner import EDMConfig
        _edm_cfg_raw = CONFIG.diffusion.get("edm", {})
        _edm_config = EDMConfig.from_yaml_dict(_edm_cfg_raw)

    diffusion = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=int(CONFIG.diffusion.conditioning_dim),
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        num_diffusion_steps=int(CONFIG.diffusion.steps),
        unet_kwargs=UNET_KWARGS,
        use_gradient_checkpointing=bool(
            CONFIG.diffusion.get("use_gradient_checkpointing", False)
        ),
        scheduler_type=_scheduler_type,
        conv_padding_mode=CONFIG.diffusion.get("conv_padding_mode", "zeros"),
        anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
        edm_config=_edm_config,
        causal_concat=True,   # Two-stage causal -> concat conditioning
    ).to(DEVICE)
    print(f"   diffusion : {sum(p.numel() for p in diffusion.parameters()):,} params, "
          f"causal_concat=True, scheduler={_scheduler_type}")

    # 5. Spatial projector + HR ident head (Sprint 1/2, optional via YAML)
    use_causal_proj = bool(CONFIG.encoder.get("causal_conditioning", False))
    _spatial_target_shape = tuple(CONFIG.diffusion.get("spatial_target_shape", [6, 7]))
    if use_causal_proj:
        spatial_projector = CausalConditioningProjector(
            num_vars=num_vars,
            hidden_dim=int(CONFIG.rcn.hidden_dim),
            conditioning_dim=int(CONFIG.diffusion.conditioning_dim),
            lr_shape=lr_shape,
            target_shape=_spatial_target_shape,
            num_dag_tokens=int(CONFIG.encoder.get("num_dag_tokens", 1)),
        ).to(DEVICE)
    else:
        spatial_projector = SpatialConditioningProjector(
            num_vars=num_vars,
            hidden_dim=int(CONFIG.rcn.hidden_dim),
            conditioning_dim=int(CONFIG.diffusion.conditioning_dim),
            lr_shape=lr_shape,
            target_shape=_spatial_target_shape,
        ).to(DEVICE)

    _hr_ident_cfg = CONFIG.loss.get("hr_ident", {}) or {}
    _hr_ident_enabled = bool(_hr_ident_cfg.get("enabled", False))
    _beta_hr_ident = float(_hr_ident_cfg.get("beta", 0.0))
    hr_ident_head = None
    if _hr_ident_enabled and _beta_hr_ident > 0.0:
        hr_ident_head = HRTargetIdentifiabilityHead(
            num_vars=num_vars,
            hidden_dim=int(CONFIG.rcn.hidden_dim),
            stats=list(_hr_ident_cfg.get("stats", ["mean", "std", "p95", "p99"])),
        ).to(DEVICE)

    return {
        "encoder": encoder,
        "rcn_cell": rcn_cell,
        "rcn_runner": rcn_runner,
        "regression_head": regression_head,
        "diffusion": diffusion,
        "spatial_projector": spatial_projector,
        "hr_ident_head": hr_ident_head,
        "num_vars": num_vars,
        "edm_config": _edm_config,
        "load_audit": {},  # from-scratch -> empty audit (PC4/PC13 trivially pass)
    }


print("[Cell 4] build_fresh_stack() defined. Will be called per-seed in cell 6.")


In [ ]:
# >>> Cell 5 : Helpers import + globals (SEEDS, K9_DATES, G_phys, schedule_lambdas)
#
# Imports A1 acquis from option_c_helpers + training internals + per-epoch
# schedule_lambdas (AI Eng critical fix : the schedule MUST run per-epoch
# because train_epoch_stage1 reads only scalar lambda_l1 + dag_grad_gate_value).
import json
import math
import time
import copy
from pathlib import Path
from typing import Dict, Any, Optional

import torch
import numpy as np

# A1 acquis (Q_phys variants, projection hook, verdict tree)
from path_c_plus.scripts.option_c_helpers import (
    PATHCPLUS_HYPERPARAM_OVERRIDES,
    PC13_NEW_PARAMS_ALLOWLIST,
    compute_q_phys_binary,
    compute_q_phys_adaptive,
    compute_q_phys_continuous,
    compute_phys_mag_gained,
    compute_skeleton_f1,
    install_projection_hook,
    check_pc4_pc13_gate,
    compute_h1_verdict,
    load_noncausal_baseline_metrics,
    one_sample_t_vs_noncausal_constant,
    holm_bonferroni_h2_h5,
    stamp_option_c_json,
)

# Training internals (scalar API : schedule values must be passed per-epoch)
from st_cdgm.training.training_loop import train_epoch_stage1, train_epoch_stage2
from st_cdgm.training.two_stage import (
    freeze_stage1,
    gamma_dag_warmup,
    calibrate_sigma_data_two_stage,
    causal_ablation_check,
    precompute_stage1_outputs,
    train_epoch_stage2_cached,
)
from st_cdgm.training.physics_prior import build_physical_mask

# Per-epoch schedule (AI Eng critical fix -- see cell 2 docstring)
from scripts.finetune_stage1_bundle_b import schedule_lambdas, DEFAULT_HYPERPARAMS

# -----------------------------------------------------------------------------
# Hyperparam dict for schedule_lambdas (DEFAULT + Path C+ overrides applied)
# -----------------------------------------------------------------------------
HP_OPTION_C = copy.deepcopy(DEFAULT_HYPERPARAMS)
# Apply Path C+ overrides on top of DEFAULT_HYPERPARAMS so schedule_lambdas
# produces the Path C+ schedule (lambda_l1 0.04 -> 0.005, gate auto-scale, etc.)
HP_OPTION_C.update({
    "lambda_dag_prior": PATHCPLUS_HYPERPARAM_OVERRIDES["lambda_dag_prior"],
    "lambda_l1_start":  PATHCPLUS_HYPERPARAM_OVERRIDES["lambda_l1_start"],
    "lambda_l1_end":    PATHCPLUS_HYPERPARAM_OVERRIDES["lambda_l1_end"],
    "g_phys_alpha":     PATHCPLUS_HYPERPARAM_OVERRIDES["g_phys_alpha"],
    "dag_gate_warmup_start_epoch": PATHCPLUS_HYPERPARAM_OVERRIDES["dag_gate_warmup_start_epoch"],
    "dag_gate_warmup_end_epoch":   PATHCPLUS_HYPERPARAM_OVERRIDES["dag_gate_warmup_end_epoch"],
})
print(f"[Cell 5] HP_OPTION_C ready : lambda_dag_prior={HP_OPTION_C['lambda_dag_prior']}, "
      f"lambda_l1 schedule={HP_OPTION_C['lambda_l1_start']} -> {HP_OPTION_C['lambda_l1_end']}, "
      f"gate warmup auto-scale")

# -----------------------------------------------------------------------------
# Seeds + dates + G_phys + commit SHA
# -----------------------------------------------------------------------------
# Single seed, full two-stage (user decision). Seed 42 is the project's
# canonical seed : the A1 H1_PASS run + locked hyperparam commit 78a3783 were
# at seed 42, and every smoke / probe used it -> maximal comparability with the
# existing 6-node results. Multi-seed (PC8) can be added later by re-listing.
SEEDS = [42]

# Resolve num_vars from a probe stack to size G_phys correctly.
# We do NOT keep this stack alive -- cell 6 builds its own per-seed.
_probe_stack = build_fresh_stack(seed=42)
NUM_VARS = _probe_stack["num_vars"]
del _probe_stack
# 9-node : G_phys must use the extended labels/edges (10 physical edges incl.
# the Held-Soden wet closure W500->SP_HR, IVT->SP_HR, Q850->IVT). The 6-node
# default build_physical_mask(6) would silently mis-size / mis-label A_dag.
if EXTENDED_9NODE:
    assert NUM_VARS == 9, (
        f"EXTENDED_9NODE=True but num_vars={NUM_VARS} (expected 9). "
        f"Check builder.extended_9node + metapath injection in cell 3."
    )
    from st_cdgm.training.physics_prior import (
        VAR_LABELS_9NODE, EXPECTED_EDGES_9NODE,
    )
    G_phys = build_physical_mask(
        num_vars=9, var_labels=VAR_LABELS_9NODE, expected_edges=EXPECTED_EDGES_9NODE,
    )
    print(f"[Cell 5] 9-node G_phys labels={VAR_LABELS_9NODE}")
else:
    G_phys = build_physical_mask(num_vars=NUM_VARS)
G_phys_np = G_phys.cpu().numpy()
print(f"\n[Cell 5] G_phys built : num_vars={NUM_VARS}, "
      f"|edges|={int((G_phys != 0).sum().item())}, sum={float(G_phys.sum().item())}")

# Pre-registration commit SHA (for traceability in JSONs)
try:
    import subprocess as _sp
    PRE_REGISTRATION_COMMIT = _sp.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd="/content/climate_data" if Path("/content/climate_data").exists() else None,
    ).decode().strip()
except Exception:
    PRE_REGISTRATION_COMMIT = "unknown"
print(f"[Cell 5] PRE_REGISTRATION_COMMIT = {PRE_REGISTRATION_COMMIT}")

# Council DS #6 : PC14 #4 distinguishes pre-registration commit (current HEAD,
# captured above) from the LOCKED hyperparam commit (the SHA at which Path C+
# overrides were frozen during the A1 run, commit 78a3783). Both are stamped
# on every JSON for full audit trail.
PATHCPLUS_LOCKED_COMMIT = "78a3783"  # A1 H1_PASS_INTERVENTIONAL_ONLY commit
print(f"[Cell 5] PATHCPLUS_LOCKED_COMMIT  = {PATHCPLUS_LOCKED_COMMIT} (hyperparam freeze)")

# -----------------------------------------------------------------------------
# Atomic save helper (mirrors noncausal cell 48 _atomic_save)
# -----------------------------------------------------------------------------
def _atomic_save_pth(payload: dict, path: Path) -> None:
    """Write to tmp + fsync + os.replace + dir fsync.

    Council SRE B1 fix : Drive FUSE mount has weird semantics; without fsync
    on the tmp file BEFORE os.replace, a Colab kill 0-60s after the call can
    leave a zero-sized epoch_last.pth in the directory listing -> torch.load
    raises UnpicklingError -> resume silently falls through to "fresh run".
    Mirrors noncausal cell 48 fsync pattern (commit f76e8f5).
    """
    import tempfile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=str(path.parent))
    try:
        os.close(fd)
        torch.save(payload, tmp)
        # Fsync the tmp file body before replace (Drive durability)
        try:
            with open(tmp, "rb") as _f:
                os.fsync(_f.fileno())
        except OSError:
            pass
        os.replace(tmp, path)
        # Fsync parent directory entry so rename is durable
        try:
            _dirfd = os.open(str(path.parent), os.O_RDONLY)
            try:
                os.fsync(_dirfd)
            finally:
                os.close(_dirfd)
        except (OSError, AttributeError):
            pass  # Windows / non-POSIX -- best effort
    except Exception:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise


def _persist_state_dict(m):
    if m is None:
        return None
    base = m.module if hasattr(m, "module") and not hasattr(m, "_orig_mod") else m
    base = getattr(base, "_orig_mod", base)
    return base.state_dict()


def _persist_load_state_dict(m, sd):
    """Load state dict tolerant to shape mismatches + _orig_mod prefix (BS22-24)."""
    if m is None or sd is None:
        return
    base = m.module if hasattr(m, "module") and not hasattr(m, "_orig_mod") else m
    base = getattr(base, "_orig_mod", base)
    stripped_sd = {k.replace("_orig_mod.", ""): v for k, v in sd.items()}
    target_sd = base.state_dict()
    matched = {}
    for tk in target_sd.keys():
        norm_tk = tk.replace("_orig_mod.", "")
        if norm_tk not in stripped_sd:
            continue
        v = stripped_sd[norm_tk]
        try:
            live_shape = tuple(target_sd[tk].shape) if hasattr(target_sd[tk], "shape") else None
        except (RuntimeError, ValueError):
            live_shape = None
        ckpt_shape = tuple(v.shape) if hasattr(v, "shape") else None
        if live_shape is not None and ckpt_shape is not None and live_shape != ckpt_shape:
            continue
        matched[tk] = v
    base.load_state_dict(matched, strict=False)


print(f"\n[Cell 5] Globals ready. Cell 6 will iterate SEEDS={SEEDS} and call "
      f"build_fresh_stack(seed) then run Stage 1 + Stage 2 from-scratch.")


In [ ]:
# >>> Cell 6 : Seed training loop (Stage 1 + Stage 2 from-scratch, per-seed)
#
# For each seed in SEEDS :
#   1. Skip if oracle_full/seed_<n>/results.json AND 3 GCM aligned JSONs exist
#      (full seed completed = all eval cells have written their outputs).
#   2. Else build fresh stack (cell 4), atomic resume from epoch_last.pth if any.
#   3. Stage 1 : 15 epochs of train_epoch_stage1, per-epoch schedule_lambdas
#      produces scalar lambda_l1 + dag_grad_gate -> train_epoch_stage1 reads them.
#   4. σ_data recalibration + O3 causal-ablation gate.
#   5. Stage 2 : 200 epochs of train_epoch_stage2_cached (BS32b pre-cache pattern).
#   6. Save final A_dag + projection_log + initial A_dag to results.json.
#
# AI Eng critical fix : schedule_lambdas is called EVERY epoch with the running
# epoch_idx + S1_EPOCHS, producing the Path C+ schedule values that
# train_epoch_stage1 reads as scalars (not via CONFIG).

import gc
import time as _mtime
import json as _mjson
import torch.nn.functional as F

# =============================================================================
# Monitoring / anomaly detection (user request : detailed logs + early anomaly
# detection). Every epoch appends one structured line to {seed_dir}/
# training_log.jsonl (survives Colab disconnects -> plot the evolution offline)
# AND prints a health banner. Anomalies are flagged loudly so a bad run can be
# killed early instead of burning 30h.
# =============================================================================
_MONITOR_STATE = {}


def _safe_f(x):
    try:
        xf = float(x)
        return xf if xf == xf and abs(xf) != float("inf") else None
    except (TypeError, ValueError):
        return None


def _monitor_epoch(seed_dir, seed, phase, epoch, total, metrics, *,
                   val_mse=None, dt=None, lr=None, extra=None):
    """Append a structured JSONL record + print health banner + flag anomalies.

    Returns the record dict (with rec['anomalies'] = list of strings).
    """
    key = (str(seed), phase)
    st = _MONITOR_STATE.setdefault(
        key, {"prev_loss": None, "best_val": float("inf"),
              "val_rises": 0, "loss_rises": 0, "dt_ema": None})

    rec = {"ts": _mtime.strftime("%Y-%m-%dT%H:%M:%S"), "seed": int(seed),
           "phase": phase, "epoch": int(epoch), "total": int(total)}
    for k, v in (metrics or {}).items():
        fv = _safe_f(v)
        rec[k] = fv if fv is not None else v
    if val_mse is not None:
        rec["val_mse"] = _safe_f(val_mse)
    if dt is not None:
        rec["epoch_time_s"] = _safe_f(dt)
    if lr is not None:
        rec["lr"] = _safe_f(lr)
    if extra:
        rec.update(extra)

    # ---- anomaly detection -------------------------------------------------
    anomalies = []
    loss = rec.get("loss", rec.get("loss_diff"))
    raw_loss = (metrics or {}).get("loss", (metrics or {}).get("loss_diff"))
    if raw_loss is not None and _safe_f(raw_loss) is None:
        anomalies.append("LOSS_NONFINITE")
    a_norm = _safe_f(rec.get("a_norm_F_end"))
    if a_norm is not None and a_norm < 0.05:
        anomalies.append(f"A_DAG_COLLAPSE(norm={a_norm:.4f})")
    lf = _safe_f(loss)
    if lf is not None and st["prev_loss"] is not None:
        if lf > st["prev_loss"] * 1.15:
            st["loss_rises"] += 1
            if st["loss_rises"] >= 3:
                anomalies.append(f"LOSS_RISING_{st['loss_rises']}x")
        else:
            st["loss_rises"] = 0
    if lf is not None:
        st["prev_loss"] = lf
    vm = _safe_f(val_mse)
    if vm is not None:
        if vm < st["best_val"] - 1e-6:
            st["best_val"] = vm
            st["val_rises"] = 0
            rec["is_best_val"] = True
        else:
            st["val_rises"] += 1
            if st["val_rises"] >= 3:
                anomalies.append(f"VAL_MSE_RISING_{st['val_rises']}x")
    dtf = _safe_f(dt)
    if dtf is not None:
        if st["dt_ema"] is None:
            st["dt_ema"] = dtf
        else:
            if dtf > 2.5 * st["dt_ema"]:
                anomalies.append(f"EPOCH_SLOWDOWN({dtf:.0f}s vs ~{st['dt_ema']:.0f}s)")
            st["dt_ema"] = 0.7 * st["dt_ema"] + 0.3 * dtf
    rec["anomalies"] = anomalies

    # ---- persist (append JSONL) -------------------------------------------
    try:
        with open(Path(seed_dir) / "training_log.jsonl", "a", encoding="utf-8") as _f:
            _f.write(_mjson.dumps(rec, default=str) + "\n")
    except Exception as _e:
        print(f"   [monitor] JSONL append failed: {_e}")

    # ---- health banner -----------------------------------------------------
    gpu = ""
    if torch.cuda.is_available():
        gpu = f" | GPUpeak={torch.cuda.max_memory_allocated() / 1024 ** 3:.1f}GB"
        torch.cuda.reset_peak_memory_stats()
    _nan = lambda x: x if x is not None else float("nan")
    if phase == "stage1":
        print(f"   [stage1 {epoch}/{total}] loss={_nan(_safe_f(rec.get('loss'))):.5f} "
              f"reg={_nan(_safe_f(rec.get('loss_reg'))):.5f} "
              f"dag={_nan(_safe_f(rec.get('loss_dag'))):.5f} | "
              f"A_norm={_nan(a_norm):.4f} A_max={_nan(_safe_f(rec.get('a_max_abs_end'))):.4f} "
              f"spars={_nan(_safe_f(rec.get('a_sparsity_end'))):.3f} | "
              f"valMSE={_nan(vm):.5f}{' [BEST]' if rec.get('is_best_val') else ''} | "
              f"{_nan(dtf):.0f}s{gpu}")
    else:
        print(f"   [stage2 {epoch}/{total}] loss_diff={_nan(_safe_f(rec.get('loss_diff'))):.5f} "
              f"ema={'on' if rec.get('ema_active') else 'off'} "
              f"facl={rec.get('loss_facl')} swd={rec.get('loss_swd')} | "
              f"{_nan(dtf):.0f}s{gpu}")
    if anomalies:
        print(f"   [monitor][!!] ANOMALIES @ {phase} ep{epoch}: {', '.join(anomalies)}")
    return rec


ts_cfg = CONFIG.two_stage
S1_EPOCHS = int(ts_cfg.stage1.epochs_max)   # 15 (CorrDiff Normal V2)
S2_EPOCHS = int(ts_cfg.stage2.epochs_max)   # 200 (CorrDiff Normal V2)
GCMS = ["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]

# DAG anti-collapse args (cell 51 noncausal style; we don't load a YAML
# dag_prior here because Path C+ uses G_phys directly via lambda_dag_prior).
_abort_on_collapse = bool(ts_cfg.stage1.get("abort_on_collapse", True))
_collapse_threshold = float(ts_cfg.stage1.get("collapse_threshold", 0.05))
_dag_floor_projection = bool(ts_cfg.stage1.get("dag_floor_projection", True))
_dag_floor_min_norm = float(ts_cfg.stage1.get("dag_floor_min_norm", 0.10))


def _seed_dir(seed: int) -> Path:
    return ORACLE_FULL_DIR / f"seed_{seed}"


def _seed_already_complete(seed: int) -> bool:
    """A seed is COMPLETE only when results.json + 3 GCM aligned JSONs exist.
    Otherwise the eval cells 8-11 still need to run."""
    d = _seed_dir(seed)
    if not (d / "results.json").exists():
        return False
    for gcm in GCMS:
        if not (d / f"aligned_metrics_{gcm}_causal.json").exists():
            return False
    return True


def _make_ckpt_payload(*, stack, optimizer, ema_diffusion, epoch_done, history,
                       stage1_epoch_done, stage2_epoch_done, sigma_data, sigma_min,
                       ablation_passed, best_val_loss, A_dag_initial, projection_log):
    payload = {
        "schema_version": 1,
        "epoch": int(epoch_done),
        "epochs_total": int(S1_EPOCHS + S2_EPOCHS),
        "encoder_state_dict": _persist_state_dict(stack["encoder"]),
        "rcn_cell_state_dict": _persist_state_dict(stack["rcn_cell"]),
        "regression_head_state_dict": _persist_state_dict(stack["regression_head"]),
        "diffusion_state_dict": _persist_state_dict(stack["diffusion"]),
        "spatial_projector_state_dict": _persist_state_dict(stack["spatial_projector"]),
        "hr_ident_head_state_dict": _persist_state_dict(stack["hr_ident_head"]),
        "diffusion_ema_state_dict": _persist_state_dict(ema_diffusion) if ema_diffusion is not None else None,
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "history": history,
        "best_val_loss": float(best_val_loss) if best_val_loss is not None else None,
        "two_stage": {
            "stage1_epoch_done": int(stage1_epoch_done),
            "stage2_epoch_done": int(stage2_epoch_done),
            "sigma_data": float(sigma_data) if sigma_data is not None else None,
            "sigma_min": float(sigma_min) if sigma_min is not None else None,
            "ablation_passed": bool(ablation_passed),
            "run_variant": "causal",
        },
        # Path C+ specific tracking
        "A_dag_initial": A_dag_initial.tolist() if A_dag_initial is not None else None,
        "projection_log_pre_spectral_lengths": [len(projection_log.get("pre_spectral_A_dag", []))],
        "rng_torch_cpu": torch.get_rng_state(),
        "rng_torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "rng_numpy": np.random.get_state(),
    }
    return payload


def _resume_from_ckpt(seed_dir: Path, stack):
    """Returns dict with epoch counters (or None for fresh run)."""
    ck_path = seed_dir / "epoch_last.pth"
    if not ck_path.exists():
        return None
    print(f"   [resume] loading {ck_path}")
    try:
        ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    except Exception as e:
        print(f"   [resume] load FAILED ({e}) -- fresh run")
        return None
    _persist_load_state_dict(stack["encoder"], ck.get("encoder_state_dict"))
    _persist_load_state_dict(stack["rcn_cell"], ck.get("rcn_cell_state_dict"))
    _persist_load_state_dict(stack["regression_head"], ck.get("regression_head_state_dict"))
    _persist_load_state_dict(stack["diffusion"], ck.get("diffusion_state_dict"))
    _persist_load_state_dict(stack["spatial_projector"], ck.get("spatial_projector_state_dict"))
    if stack.get("hr_ident_head") is not None:
        _persist_load_state_dict(stack["hr_ident_head"], ck.get("hr_ident_head_state_dict"))
    ts = ck.get("two_stage", {}) or {}
    if ck.get("rng_torch_cpu") is not None:
        try:
            torch.set_rng_state(ck["rng_torch_cpu"])
        except TypeError:
            torch.set_rng_state(torch.as_tensor(ck["rng_torch_cpu"], dtype=torch.uint8).cpu())
    if torch.cuda.is_available() and ck.get("rng_torch_cuda") is not None:
        try:
            torch.cuda.set_rng_state_all(ck["rng_torch_cuda"])
        except (TypeError, RuntimeError):
            pass
    if ck.get("rng_numpy") is not None:
        np.random.set_state(ck["rng_numpy"])
    return {
        "stage1_epoch_done": int(ts.get("stage1_epoch_done", 0)),
        "stage2_epoch_done": int(ts.get("stage2_epoch_done", 0)),
        "sigma_data": ts.get("sigma_data"),
        "sigma_min": ts.get("sigma_min"),
        "ablation_passed": bool(ts.get("ablation_passed", False)),
        "history": ck.get("history") or {},
        "best_val_loss": ck.get("best_val_loss"),
        "ema_state_dict": ck.get("diffusion_ema_state_dict"),
        "optimizer_state_dict": ck.get("optimizer_state_dict"),
        "A_dag_initial": ck.get("A_dag_initial"),
    }


# -----------------------------------------------------------------------------
# STAGE 1 real gate : deterministic F1@p99 / tail-R2 on validation
# -----------------------------------------------------------------------------
@torch.no_grad()
def _stage1_gate_eval(encoder, rcn_runner, regression_head, builder, dataloader,
                      device, tail_pct=99.0, max_batches=None):
    """Deterministic Stage-1 precip gate (no diffusion).

    Stage 1 predicts the log1p residual mu_HR ; the deterministic precip
    estimate is expm1(baseline_log + mu_HR), observed = expm1(baseline_log +
    residual). Absolute F1@p99 is LOW by construction -- the diffusion Stage 2
    is what synthesises extremes. This gate is meaningful RELATIVE to the
    6-node Stage-1 baseline trained with the SAME code (cell 6b compares them).
    """
    encoder.eval(); rcn_runner.cell.eval(); regression_head.eval()
    obs_all, pred_all, robs_all, rpred_all = [], [], [], []
    nb = 0
    for converted in iterate_batches(dataloader, builder, device):
        for batch in converted:
            tgt = batch["residual"][-1].to(device)
            if tgt.dim() == 3:
                tgt = tgt.unsqueeze(0)
            base = batch.get("baseline")
            if base is None:
                continue
            base = base[-1].to(device)
            if base.dim() == 3:
                base = base.unsqueeze(0)
            H_init = encoder.init_state(batch["hetero"]).to(device)
            drivers = [batch["lr"][t].to(device) for t in range(batch["lr"].shape[0])]
            seq_out = rcn_runner.run(H_init, drivers, reconstruction_sources=None)
            mu_HR = regression_head(seq_out.states[-1])
            if mu_HR.shape[-2:] != tgt.shape[-2:]:
                mu_HR = F.interpolate(mu_HR, size=tgt.shape[-2:], mode="bilinear", align_corners=False)
            if base.shape[-2:] != tgt.shape[-2:]:
                base = F.interpolate(base, size=tgt.shape[-2:], mode="bilinear", align_corners=False)
            m = torch.isfinite(tgt) & torch.isfinite(mu_HR) & torch.isfinite(base)
            if not m.any():
                continue
            precip_obs = torch.expm1(base + tgt)
            precip_hat = torch.expm1(base + mu_HR)
            obs_all.append(precip_obs[m].float().cpu().numpy().ravel())
            pred_all.append(precip_hat[m].float().cpu().numpy().ravel())
            robs_all.append(tgt[m].float().cpu().numpy().ravel())
            rpred_all.append(mu_HR[m].float().cpu().numpy().ravel())
            nb += 1
            if max_batches and nb >= max_batches:
                break
        if max_batches and nb >= max_batches:
            break
    if not obs_all:
        return {"status": "no_data"}
    obs = np.concatenate(obs_all); pred = np.concatenate(pred_all)
    r_obs = np.concatenate(robs_all); r_pred = np.concatenate(rpred_all)
    thr = float(np.percentile(obs, tail_pct))
    yo = obs >= thr; yp = pred >= thr
    tp = int(np.sum(yo & yp)); fp = int(np.sum(~yo & yp)); fn = int(np.sum(yo & ~yp))
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    rmse = float(np.sqrt(np.mean((pred - obs) ** 2)))
    pear = (float(np.corrcoef(pred, obs)[0, 1])
            if obs.std() > 0 and pred.std() > 0 else float("nan"))
    if int(yo.sum()) > 2:
        ss_res = float(np.sum((obs[yo] - pred[yo]) ** 2))
        ss_tot = float(np.sum((obs[yo] - obs[yo].mean()) ** 2))
        tail_r2 = (1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    else:
        tail_r2 = float("nan")
    return {
        "status": "ok",
        "f1_at_p99": float(f1), "precision_at_p99": float(prec), "recall_at_p99": float(rec),
        "p99_threshold_mm": thr, "rmse": rmse, "pearson": pear, "tail_r2": float(tail_r2),
        "residual_rmse": float(np.sqrt(np.mean((r_pred - r_obs) ** 2))),
        "n_pixels": int(obs.size), "n_tail": int(yo.sum()),
    }


# -----------------------------------------------------------------------------
# Main per-seed loop
# -----------------------------------------------------------------------------
def train_one_seed(seed: int):
    print("\n" + "=" * 78)
    print(f"[SEED {seed}] Option C training (Stage 1 = {S1_EPOCHS} ep, "
          f"Stage 2 = {S2_EPOCHS} ep)")
    print("=" * 78)
    seed_dir = _seed_dir(seed)
    seed_dir.mkdir(parents=True, exist_ok=True)

    # Council SRE O4 fix : orphan tmp cleanup from killed sessions
    for _orphan in list(seed_dir.glob("epoch_last.pth.*.tmp")):
        try:
            _orphan.unlink()
            print(f"   [cleanup] removed orphan tmp: {_orphan.name}")
        except OSError:
            pass

    # Council AI Eng B3 fix : rebuild train_dataset per seed (IterableDataset
    # single-shot exhaustion when running multiple seeds in one kernel).
    train_dataset_seed = pipeline.build_sequence_dataset(
        split="train", seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True,
    )

    # Build fresh stack (resets RNG to `seed`)
    stack = build_fresh_stack(seed)
    encoder = stack["encoder"]
    rcn_cell = stack["rcn_cell"]
    rcn_runner = stack["rcn_runner"]
    regression_head = stack["regression_head"]
    diffusion = stack["diffusion"]
    spatial_projector = stack["spatial_projector"]
    hr_ident_head = stack["hr_ident_head"]
    edm_config_init = stack["edm_config"]

    # PC4+PC13 gate -- trivially passes for from-scratch
    pc4_ok, pc4_excl, pc13_evidence = check_pc4_pc13_gate(stack["load_audit"])
    assert pc4_ok, f"PC4 fail (should not happen for from-scratch): {pc4_excl}"

    # Install 3-point projection hook on rcn_cell (A1 acquis)
    projection_log = install_projection_hook(rcn_cell)

    # Resume? (state -> stack, return counters)
    resume = _resume_from_ckpt(seed_dir, stack)
    if resume is None:
        s1_from = 0
        s2_from = 0
        skip_calib = False
        saved_sigma_data = None
        saved_sigma_min = None
        skip_ablation = False
        history = {k: [] for k in ["val_mse_stage1", "loss_diff_train",
                                    "epoch_time", "contrastive_dag", "dag_sensitivity",
                                    "contrastive_lambda", "contrastive_mode"]}
        best_s1_val = math.inf
        A_dag_initial = rcn_cell.A_dag.detach().cpu().numpy().copy()
        ema_resume_sd = None
        optim_resume_sd = None
    else:
        s1_from = resume["stage1_epoch_done"]
        s2_from = resume["stage2_epoch_done"]
        saved_sigma_data = resume["sigma_data"]
        saved_sigma_min = resume["sigma_min"]
        skip_calib = saved_sigma_data is not None
        skip_ablation = resume["ablation_passed"]
        history = resume["history"] or {k: [] for k in
            ["val_mse_stage1", "loss_diff_train", "epoch_time",
             "contrastive_dag", "dag_sensitivity",
             "contrastive_lambda", "contrastive_mode"]}
        best_s1_val = resume["best_val_loss"] if resume["best_val_loss"] is not None else math.inf
        A_dag_initial = (np.asarray(resume["A_dag_initial"])
                          if resume["A_dag_initial"] is not None
                          else rcn_cell.A_dag.detach().cpu().numpy().copy())
        ema_resume_sd = resume["ema_state_dict"]
        optim_resume_sd = resume["optimizer_state_dict"]
        print(f"   [resume] s1_done={s1_from}/{S1_EPOCHS}, s2_done={s2_from}/{S2_EPOCHS}, "
              f"sigma_data={saved_sigma_data}, ablation_passed={skip_ablation}")

    # -------------------------------------------------------------------------
    # Materialize lazy params (cell 51 BS6 pattern)
    # -------------------------------------------------------------------------
    if s1_from == 0:
        print("   [Stage 1] materializing lazy SAGEConv params...")
        encoder.train(); rcn_cell.train(); regression_head.train()
        for _conv in iterate_batches(train_dataloader, builder, DEVICE):
            for _b in _conv:
                with torch.no_grad():
                    _H = encoder.init_state(_b["hetero"]).to(DEVICE)
                    _lr = _b["lr"].to(DEVICE)
                    _drv = [_lr[t] for t in range(_lr.shape[0])]
                    _seq = rcn_runner.run(_H, _drv, reconstruction_sources=None)
                    _ = regression_head(_seq.states[-1])
                break
            break

    # -------------------------------------------------------------------------
    # Stage 1 optimizer
    # -------------------------------------------------------------------------
    stage1_params = (
        list(encoder.parameters())
        + list(rcn_cell.parameters())
        + list(regression_head.parameters())
    )
    optimizer_s1 = torch.optim.AdamW(
        stage1_params,
        lr=float(ts_cfg.stage1.lr),
        weight_decay=float(ts_cfg.stage1.weight_decay),
    )
    if optim_resume_sd is not None and s2_from == 0 and s1_from > 0:
        try:
            optimizer_s1.load_state_dict(optim_resume_sd)
            print("   [resume] optimizer_s1 state restored")
        except Exception as e:
            print(f"   [resume] optimizer_s1 restore failed: {e}")

    val_mse_history = list(history.get("val_mse_stage1", []))
    patience = int(ts_cfg.stage1.early_stop_patience)
    no_improve_s1 = 0
    best_s1_epoch = 0

    # ============================ STAGE 1 ===================================
    if s1_from >= S1_EPOCHS:
        print(f"   [Stage 1] already done ({s1_from}/{S1_EPOCHS}) -- skip")
    else:
        print(f"\n   [Stage 1] starting at epoch {s1_from + 1}/{S1_EPOCHS}")
        for s1_epoch in range(s1_from, S1_EPOCHS):
            # AI Eng CRITICAL FIX : per-epoch schedule_lambdas
            sched = schedule_lambdas(s1_epoch, S1_EPOCHS, HP_OPTION_C)
            print(f"\n   --- Stage 1 epoch {s1_epoch + 1}/{S1_EPOCHS} | "
                  f"lambda_l1={sched['lambda_l1']:.4f} | "
                  f"gate={sched['dag_grad_gate']:.2f} | "
                  f"gamma_dag={sched['gamma_dag']:.4f} ---")
            _t_s1 = time.time()

            s1_metrics = train_epoch_stage1(
                encoder=encoder,
                rcn_runner=rcn_runner,
                regression_head=regression_head,
                optimizer=optimizer_s1,
                data_loader=iterate_batches(train_dataloader, builder, DEVICE),
                device=DEVICE,
                epoch_idx=s1_epoch,
                lambda_reg=float(ts_cfg.stage1.lambda_reg),
                beta_rec=float(ts_cfg.stage1.beta_rec),
                gamma_dag_max=float(ts_cfg.stage1.gamma_dag_max),
                gamma_dag_warmup_epochs=int(ts_cfg.stage1.gamma_dag_warmup_epochs),
                # AI Eng fix: scalar from schedule_lambdas (per-epoch annealing)
                lambda_l1=float(sched["lambda_l1"]),
                # AI Eng risk #2 : use sched value (consistent with HP_OPTION_C)
                lambda_dag_prior=float(sched["lambda_dag_prior"]),
                dag_prior=G_phys,                       # physics prior matrix
                dag_grad_gate_value=float(sched["dag_grad_gate"]),
                abort_on_collapse=_abort_on_collapse,
                collapse_threshold=_collapse_threshold,
                dag_floor_projection=_dag_floor_projection,
                dag_floor_min_norm=_dag_floor_min_norm,
                gradient_clipping=CONFIG.training.gradient_clipping,
                log_interval=int(CONFIG.training.log_every),
                use_amp=bool(CONFIG.training.get("use_amp", True)),
            )

            # Quick val MSE (cell 51 noncausal style)
            encoder.eval(); rcn_runner.cell.eval(); regression_head.eval()
            val_losses = []
            with torch.no_grad():
                for converted in iterate_batches(val_dataloader, builder, DEVICE):
                    for batch in converted:
                        target_residual = batch["residual"][-1].to(DEVICE)
                        if target_residual.dim() == 3:
                            target_residual = target_residual.unsqueeze(0)
                        H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
                        drivers = [batch["lr"][t].to(DEVICE) for t in range(batch["lr"].shape[0])]
                        seq_out = rcn_runner.run(H_init, drivers, reconstruction_sources=None)
                        mu_HR = regression_head(seq_out.states[-1])
                        if mu_HR.shape != target_residual.shape:
                            mu_HR = F.interpolate(mu_HR, size=target_residual.shape[-2:],
                                                  mode="bilinear", align_corners=False)
                        mask = torch.isfinite(target_residual) & torch.isfinite(mu_HR)
                        if mask.any():
                            sq = (torch.where(mask, mu_HR, torch.zeros_like(mu_HR))
                                  - torch.where(mask, target_residual, torch.zeros_like(target_residual))) ** 2
                            v = float((sq * mask.float()).sum().item() / max(1.0, mask.float().sum().item()))
                            if v == v:  # not NaN
                                val_losses.append(v)
            val_mse = float(np.mean(val_losses)) if val_losses else float("nan")
            val_mse_history.append(val_mse)
            _monitor_epoch(
                seed_dir, seed, "stage1", s1_epoch + 1, S1_EPOCHS, s1_metrics,
                val_mse=val_mse, dt=time.time() - _t_s1,
                lr=optimizer_s1.param_groups[0]["lr"],
                extra={"lambda_l1": float(sched["lambda_l1"]),
                       "dag_grad_gate": float(sched["dag_grad_gate"]),
                       "gamma_dag": float(sched["gamma_dag"])},
            )

            # Atomic persist per epoch
            history["val_mse_stage1"] = val_mse_history
            payload = _make_ckpt_payload(
                stack=stack, optimizer=optimizer_s1, ema_diffusion=None,
                epoch_done=s1_epoch + 1, history=history,
                stage1_epoch_done=s1_epoch + 1, stage2_epoch_done=0,
                sigma_data=None, sigma_min=None,
                ablation_passed=False,
                best_val_loss=best_s1_val,
                A_dag_initial=A_dag_initial,
                projection_log=projection_log,
            )
            _atomic_save_pth(payload, seed_dir / "epoch_last.pth")

            if val_mse < best_s1_val - 1e-5:
                best_s1_val = val_mse
                best_s1_epoch = s1_epoch + 1
                no_improve_s1 = 0
                try:
                    _atomic_save_pth(payload, seed_dir / "epoch_best_stage1.pth")
                    print(f"     [ckpt] new best Stage-1 val MSE={best_s1_val:.5f} "
                          f"@ep{best_s1_epoch} -> epoch_best_stage1.pth")
                except Exception as _ckpt_e:
                    print(f"     [ckpt] best save failed: {_ckpt_e}")
            else:
                no_improve_s1 += 1
            if no_improve_s1 >= patience:
                print(f"   [Stage 1] early stop @ ep{s1_epoch + 1} "
                      f"(best={best_s1_val:.5f}@{best_s1_epoch})")
                break

    # -------------------------------------------------------------------------
    # STAGE 1 real gate : skip sigma recalib + O3 ablation + Stage 2 entirely.
    # Compute the deterministic precip gate, persist stage1_gate.json (NOT
    # results.json, so the heavy eval is not triggered), and early-return.
    # -------------------------------------------------------------------------
    if STAGE1_ONLY:
        print(f"\n   [STAGE1_ONLY] computing deterministic Stage-1 gate "
              f"(F1@p99 / tail-R2 / RMSE / Pearson) on validation...")
        gate = _stage1_gate_eval(
            encoder, rcn_runner, regression_head, builder, val_dataloader, DEVICE,
            tail_pct=99.0,
        )
        print(f"   [STAGE1_ONLY gate] {gate}")
        A_dag_final_np = rcn_cell.A_dag.detach().cpu().numpy().copy()
        q_phys_binary, _, _, n_extra_bin = compute_q_phys_binary(A_dag_final_np, G_phys_np)
        q_phys_cont, collapsed = compute_q_phys_continuous(A_dag_final_np, G_phys_np)
        phys_mag_gained = compute_phys_mag_gained(A_dag_final_np, A_dag_initial, G_phys_np)
        gate_results = {
            "seed": int(seed),
            "stage1_only": True,
            "extended_9node": bool(EXTENDED_9NODE),
            "num_vars": int(NUM_VARS),
            "stage1_gate": gate,
            "best_s1_val_mse": float(best_s1_val) if math.isfinite(best_s1_val) else None,
            "A_dag_initial": A_dag_initial.tolist(),
            "A_dag_final": A_dag_final_np.tolist(),
            "q_phys_binary_final": q_phys_binary,
            "q_phys_continuous_final": q_phys_cont,
            "q_phys_collapsed_final": bool(collapsed),
            "phys_mag_gained_final": phys_mag_gained,
            "n_extra_edges_final": int(n_extra_bin),
        }
        gate_results = stamp_option_c_json(
            gate_results, seed=seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
            pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
            pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=False,
        )
        (seed_dir / "stage1_gate.json").write_text(json.dumps(gate_results, indent=2, default=str))
        print(f"   [STAGE1_ONLY] saved -> {seed_dir / 'stage1_gate.json'}  | "
              f"F1@p99={gate.get('f1_at_p99')}, tail_r2={gate.get('tail_r2')}, "
              f"Q_phys_cont={q_phys_cont:.4f}")
        return {
            "stack": stack, "ema_diffusion": None,
            "ablation_report": {"skipped_stage1_only": True},
            "seed_dir": seed_dir, "A_dag_initial": A_dag_initial,
            "A_dag_final": A_dag_final_np, "projection_log": projection_log,
            "stage1_gate": gate,
        }

    # -------------------------------------------------------------------------
    # σ_data recalibration + O3 ablation gate
    # -------------------------------------------------------------------------
    if skip_calib and saved_sigma_data is not None:
        new_sigma_data = float(saved_sigma_data)
        new_sigma_min = float(saved_sigma_min if saved_sigma_min is not None
                              else max(1e-4, new_sigma_data * float(ts_cfg.stage2.sigma_min_scale_factor)))
        print(f"\n   [σ recalib] skipped (resumed) -- sigma_data={new_sigma_data:.5f}")
    else:
        print("\n   [σ recalib] computing post-Stage-1 sigma_data...")
        calib = calibrate_sigma_data_two_stage(
            encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
            data_loader=train_dataloader,
            iterate_batches_fn=iterate_batches,
            builder=builder, device=DEVICE, max_samples=200,
        )
        new_sigma_data = float(calib["sigma_data"])
        new_sigma_min = max(1e-4, new_sigma_data * float(ts_cfg.stage2.sigma_min_scale_factor))
        print(f"     new sigma_data={new_sigma_data:.5f}, sigma_min={new_sigma_min:.5f}")

    CONFIG.diffusion.edm.sigma_data = new_sigma_data
    CONFIG.diffusion.edm.sigma_min = new_sigma_min
    from st_cdgm.models.edm_preconditioner import EDMConfig as _EDMConfig
    diffusion.edm_config = _EDMConfig(
        sigma_data=new_sigma_data, sigma_min=new_sigma_min,
        sigma_max=float(CONFIG.diffusion.edm.sigma_max),
        rho=float(CONFIG.diffusion.edm.rho),
        P_mean=float(CONFIG.diffusion.edm.P_mean),
        P_std=float(CONFIG.diffusion.edm.P_std),
    )

    if skip_ablation:
        ablation_report = {"passes": True, "ratio": 1.0, "threshold": float(ts_cfg.causal_ablation.threshold)}
        print("   [O3 gate] skipped (resumed, previously passed)")
    else:
        print("   [O3 gate] running causal_ablation_check...")
        ablation_report = causal_ablation_check(
            encoder=encoder, rcn_runner=rcn_runner,
            rcn_cell=rcn_cell.module if hasattr(rcn_cell, "module") else rcn_cell,
            regression_head=regression_head,
            data_loader=val_dataloader,
            iterate_batches_fn=iterate_batches,
            builder=builder, device=DEVICE,
            n_samples=int(ts_cfg.causal_ablation.n_samples),
            threshold=float(ts_cfg.causal_ablation.threshold),
        )
        if not ablation_report["passes"] and bool(ts_cfg.causal_ablation.abort_if_fail):
            raise RuntimeError(
                f"[SEED {seed}] Causal ablation FAILED "
                f"(ratio={ablation_report['ratio']:.4f} < {ablation_report['threshold']}). "
                "DAG decorative -- abort Stage 2."
            )

    # -------------------------------------------------------------------------
    # Freeze Stage 1 + build Stage 2 optimizer + EMA + BS32b cache
    # -------------------------------------------------------------------------
    freeze_stage1(encoder, rcn_runner.cell, regression_head)
    optimizer_s2 = torch.optim.AdamW(
        diffusion.parameters(), lr=float(ts_cfg.stage2.lr), weight_decay=1e-4,
    )
    if optim_resume_sd is not None and s2_from > 0:
        try:
            optimizer_s2.load_state_dict(optim_resume_sd)
            print("   [resume] optimizer_s2 state restored")
        except Exception as e:
            print(f"   [resume] optimizer_s2 restore failed: {e}")

    # EMA
    _ema_cfg = (ts_cfg.stage2 or {}).get("ema", {}) or {}
    _ema_enabled = bool(_ema_cfg.get("enabled", False))
    _ema_decay = float(_ema_cfg.get("decay", 0.9999))
    ema_diffusion = None
    if _ema_enabled:
        ema_diffusion = copy.deepcopy(diffusion).eval()
        for p in ema_diffusion.parameters():
            p.requires_grad_(False)
        if ema_resume_sd is not None:
            try:
                ema_diffusion.load_state_dict(ema_resume_sd)
                print("   [resume] EMA diffusion state restored")
            except Exception as e:
                print(f"   [resume] EMA restore failed ({e}) -- fresh EMA from live weights")

    # BS32b cache (per-seed cache file so seeds don't collide)
    bs32b_cache_path = seed_dir / "stage1_cache.pt"
    # Council SRE B3 fix : drop stale cache if Stage 1 not certified done.
    # Otherwise Stage 2 would train on mu_HR / baseline_log from a partial Stage 1.
    if s1_from < S1_EPOCHS and bs32b_cache_path.exists():
        print(f"   [BS32b] Stage 1 partial (s1_from={s1_from}/{S1_EPOCHS}) "
              f"-- DROPPING stale cache {bs32b_cache_path.name}")
        try:
            bs32b_cache_path.unlink()
        except OSError as e:
            print(f"   [BS32b] cache unlink failed ({e})")
    _existing_cache = None
    if bs32b_cache_path.exists():
        print(f"   [BS32b] cache found at {bs32b_cache_path}, loading...")
        _existing_cache = torch.load(bs32b_cache_path, map_location="cpu", weights_only=False)

    # AI Eng B3 : use the per-seed rebuilt dataset
    bs32b_cache = precompute_stage1_outputs(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        train_dataset=train_dataset_seed,
        iterate_batches_fn=lambda s: convert_sample_to_batch(s, builder, DEVICE),
        device=DEVICE,
        dag_variants=["normal"],
        existing_cache=_existing_cache,
    )
    try:
        torch.save(bs32b_cache, bs32b_cache_path)
        print(f"   [BS32b] cache saved : {bs32b_cache_path}")
    except Exception as e:
        print(f"   [BS32b] cache save failed: {e}")

    from torch.utils.data import DataLoader as _DL2, Dataset as _DS2

    class _BS32bDataset(_DS2):
        def __init__(self, cache):
            self.mu = cache["mu_HR"]; self.base = cache["baseline_log"]
            self.delta = cache["delta_target"]; self.mask = cache["valid_mask"]
        def __len__(self):
            return self.mu.shape[0]
        def __getitem__(self, idx):
            return {"mu_HR": self.mu[idx], "baseline_log": self.base[idx],
                    "delta_target": self.delta[idx], "valid_mask": self.mask[idx]}

    cached_dataloader = _DL2(
        _BS32bDataset(bs32b_cache), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=PIN_MEMORY, drop_last=False,
    )

    # ============================ STAGE 2 ===================================
    if s2_from >= S2_EPOCHS:
        print(f"\n   [Stage 2] already done ({s2_from}/{S2_EPOCHS}) -- skip")
    else:
        print(f"\n   [Stage 2] starting at epoch {s2_from + 1}/{S2_EPOCHS}")
        for s2_epoch in range(s2_from, S2_EPOCHS):
            t0 = time.time()
            s2_metrics = train_epoch_stage2_cached(
                diffusion_decoder=diffusion,
                optimizer=optimizer_s2,
                cached_dataloader=cached_dataloader,
                device=DEVICE,
                use_amp=bool(CONFIG.training.get("use_amp", True)),
                gradient_clipping=CONFIG.training.gradient_clipping,
                log_every=int(CONFIG.training.get("log_every", 20)),
                lambda_contrastive_dag=0.0,   # noncausal-style cached step, no contrastive
                ema_model=ema_diffusion,
                ema_decay=_ema_decay,
                log_loss_components=True,
            )
            dt = time.time() - t0
            history.setdefault("loss_diff_train", []).append(s2_metrics["loss_diff"])
            history.setdefault("epoch_time", []).append(dt)
            _monitor_epoch(
                seed_dir, seed, "stage2", s2_epoch + 1, S2_EPOCHS, s2_metrics,
                dt=dt, lr=optimizer_s2.param_groups[0]["lr"],
            )

            payload = _make_ckpt_payload(
                stack=stack, optimizer=optimizer_s2, ema_diffusion=ema_diffusion,
                epoch_done=S1_EPOCHS + s2_epoch + 1, history=history,
                stage1_epoch_done=S1_EPOCHS, stage2_epoch_done=s2_epoch + 1,
                sigma_data=new_sigma_data, sigma_min=new_sigma_min,
                ablation_passed=bool(ablation_report.get("passes", False)),
                best_val_loss=best_s1_val,
                A_dag_initial=A_dag_initial,
                projection_log=projection_log,
            )
            _atomic_save_pth(payload, seed_dir / "epoch_last.pth")

    # -------------------------------------------------------------------------
    # Final results.json (Path C+ specific : Q_phys + projection_log + A_dag)
    # -------------------------------------------------------------------------
    A_dag_final_np = rcn_cell.A_dag.detach().cpu().numpy().copy()
    q_phys_binary, _, _, n_extra_bin = compute_q_phys_binary(A_dag_final_np, G_phys_np)
    q_phys_adaptive, _, _, _thr_a = compute_q_phys_adaptive(A_dag_final_np, G_phys_np)
    q_phys_cont, collapsed = compute_q_phys_continuous(A_dag_final_np, G_phys_np)
    phys_mag_gained = compute_phys_mag_gained(A_dag_final_np, A_dag_initial, G_phys_np)
    skel_f1, _thr_s = compute_skeleton_f1(A_dag_final_np, G_phys_np)

    results = {
        "seed": int(seed),
        "ablation_report": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                             for k, v in ablation_report.items()},
        "sigma_data_post_s1": float(new_sigma_data),
        "sigma_min_post_s1": float(new_sigma_min),
        "best_s1_val_mse": float(best_s1_val) if math.isfinite(best_s1_val) else None,
        "A_dag_initial": A_dag_initial.tolist(),
        "A_dag_final":   A_dag_final_np.tolist(),
        "q_phys_binary_final":   q_phys_binary,
        "q_phys_adaptive_final": q_phys_adaptive,
        "q_phys_continuous_final": q_phys_cont,
        "q_phys_collapsed_final": bool(collapsed),
        "phys_mag_gained_final": phys_mag_gained,
        "skeleton_f1_final": skel_f1,
        "n_extra_edges_final": int(n_extra_bin),
        "projection_log_summary": {
            "n_pre_spectral":  len(projection_log.get("pre_spectral_A_dag", [])),
            "n_post_spectral": len(projection_log.get("post_spectral_A_dag", [])),
            "n_post_floor":    len(projection_log.get("post_floor_A_dag", [])),
        },
    }
    results = stamp_option_c_json(
        results, seed=seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
        pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
        pre_registration_commit=PRE_REGISTRATION_COMMIT,
        is_aggregate=False,
    )
    (seed_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))
    print(f"\n   [SEED {seed}] results.json saved : Q_phys_cont={q_phys_cont:.4f} "
          f"(binary={q_phys_binary:.4f}, adaptive={q_phys_adaptive:.4f}, "
          f"phys_mag_gained={phys_mag_gained:.4f}, skel_f1={skel_f1:.4f}, "
          f"n_extra={n_extra_bin})")

    # Return live stack so eval cells 7-11 can use it without re-loading
    return {
        "stack": stack,
        "ema_diffusion": ema_diffusion,
        "ablation_report": ablation_report,
        "seed_dir": seed_dir,
        "A_dag_initial": A_dag_initial,
        "A_dag_final": A_dag_final_np,
        "projection_log": projection_log,
    }


# -----------------------------------------------------------------------------
# Master loop : per-seed train + eval with SPLIT skip-gates
#
# Council SRE B2 : two distinct skip-gates.
#   - train_done = results.json exists  (don't re-train)
#   - eval_done  = 5 eval files exist   (don't re-eval)
#
# Council AI Eng B2 : run `run_full_eval_for_seed` INSIDE this loop after
# training. Kernel kill between seeds preserves done-seeds' eval JSONs.
#
# Council SRE B4 : try/except around train_one_seed -- one seed failing does
# NOT kill the whole 3-seed run. Tombstone JSON written, loop continues.
# -----------------------------------------------------------------------------
COMPLETED_SEEDS = []
TRAINED_SEEDS = []
FAILED_SEEDS = {}   # seed -> error message

def _maybe_eval(seed):
    """Forward-call to cell 8's run_full_eval_for_seed if it exists."""
    if STAGE1_ONLY:
        print(f"   [eval] STAGE1_ONLY -- skipping full two-stage eval "
              f"(gate lives in seed_{seed}/stage1_gate.json)")
        return
    fn = globals().get("run_full_eval_for_seed")
    if fn is None:
        print(f"   [eval] cell 8 not yet defined -- deferring eval")
        return
    try:
        fn(seed)
    except Exception as _eval_err:
        import traceback as _tb
        print(f"   [eval] seed {seed} eval FAILED : {_eval_err}")
        (_seed_dir(seed) / "EVAL_FAILED.json").write_text(json.dumps({
            "seed": seed, "error": str(_eval_err), "traceback": _tb.format_exc()[-2000:],
        }, indent=2, default=str))

for _seed in SEEDS:
    seed_dir = _seed_dir(_seed)
    train_done = (seed_dir / "results.json").exists()
    eval_done = _seed_already_complete(_seed)

    if eval_done:
        print(f"[SEED {_seed}] FULLY COMPLETE -- skip")
        COMPLETED_SEEDS.append(_seed)
        TRAINED_SEEDS.append(_seed)
        continue

    if train_done:
        print(f"[SEED {_seed}] training DONE -- eval only")
        TRAINED_SEEDS.append(_seed)
        _maybe_eval(_seed)
        if _seed_already_complete(_seed):
            COMPLETED_SEEDS.append(_seed)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    try:
        train_one_seed(_seed)
        TRAINED_SEEDS.append(_seed)
    except Exception as _train_err:
        import traceback
        _tb = traceback.format_exc()
        print(f"[SEED {_seed}] TRAINING FAILED : {_train_err}")
        FAILED_SEEDS[_seed] = str(_train_err)
        (seed_dir / "PROTOCOL_FAILURE.json").write_text(json.dumps({
            "seed": _seed,
            "verdict": "PROTOCOL_FAILURE",
            "error": str(_train_err),
            "traceback": _tb[-2000:],
            "stage": "training",
        }, indent=2, default=str))
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    _maybe_eval(_seed)
    if _seed_already_complete(_seed):
        COMPLETED_SEEDS.append(_seed)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n[Cell 6] seed loop done.")
print(f"   TRAINED_SEEDS   = {TRAINED_SEEDS}")
print(f"   COMPLETED_SEEDS = {COMPLETED_SEEDS}  (train + 3 GCM eval)")
if FAILED_SEEDS:
    print(f"   FAILED_SEEDS    = {FAILED_SEEDS}")


In [ ]:
# >>> Cell 6b : STAGE-1 REAL GATE verdict -- 9-node vs 6-node (deterministic)
#
# Compares the Stage-1 precip gate (F1@p99 / tail-R2) of THIS 9-node run
# (oracle_9node/seed_*/stage1_gate.json) against the 6-node Stage-1 baseline
# (oracle_full/seed_*/stage1_gate.json). To produce the 6-node baseline gates,
# run the 6-node base notebook (st_cdgm_path_c_option_c.ipynb) with a STAGE1_ONLY
# patch, OR re-run THIS notebook once with EXTENDED_9NODE=False + STAGE1_ONLY=True.
#
# Decision rule (RELATIVE -- absolute Stage-1 F1@p99 is low by design, the
# diffusion Stage 2 is what synthesises extremes) :
#   GO    if mean(F1@p99_9) - mean(F1@p99_6) >= pooled std AND tail_R2 not worse
#   WEAK  if positive but within noise
#   NO-GO if <= 0  -> humid chain adds no Stage-1 tail skill; save Stage-2 compute
import json as _json
import numpy as _np
from pathlib import Path as _P

_ROOT = _P("/content/drive/MyDrive/climate_data")
_DIR9 = _ROOT / "oracle_9node"
_DIR6 = _ROOT / "oracle_full"


def _collect_gate(d):
    out = {}
    if not d.exists():
        return out
    for sd in sorted(d.glob("seed_*")):
        gp = sd / "stage1_gate.json"
        if gp.exists():
            try:
                g = _json.loads(gp.read_text()).get("stage1_gate", {})
                if g.get("status") == "ok":
                    out[sd.name] = g
            except Exception as e:
                print(f"  [warn] {gp}: {e}")
    return out


def _agg(gd, key):
    vals = [v[key] for v in gd.values()
            if v.get(key) is not None and v.get(key) == v.get(key)]
    if not vals:
        return float("nan"), float("nan"), 0
    return float(_np.mean(vals)), float(_np.std(vals)), len(vals)


g9 = _collect_gate(_DIR9)
g6 = _collect_gate(_DIR6)

print("=" * 78)
print("[Cell 6b] STAGE-1 REAL GATE -- 9-node vs 6-node (deterministic, no diffusion)")
print("=" * 78)
for tag, gd in (("9-node", g9), ("6-node", g6)):
    f1m, f1s, n = _agg(gd, "f1_at_p99")
    t2m, t2s, _ = _agg(gd, "tail_r2")
    rm, rs, _ = _agg(gd, "rmse")
    print(f"  {tag:7s} (n={n}) : F1@p99={f1m:.4f}+/-{f1s:.4f} | "
          f"tail_R2={t2m:.4f}+/-{t2s:.4f} | RMSE={rm:.4f}+/-{rs:.4f}")

f1_9, s1_9, n9 = _agg(g9, "f1_at_p99")
f1_6, s1_6, n6 = _agg(g6, "f1_at_p99")
t2_9, _, _ = _agg(g9, "tail_r2")
t2_6, _, _ = _agg(g6, "tail_r2")

if n9 == 0:
    verdict = "NO_9NODE_GATE -- run cell 6 with STAGE1_ONLY=True first"
elif n6 == 0:
    verdict = ("NO_6NODE_BASELINE -- produce oracle_full/seed_*/stage1_gate.json "
               "(6-node Stage-1) then re-run this cell")
else:
    delta = f1_9 - f1_6
    pooled = (s1_9 + s1_6) / 2 + 1e-9
    if (delta > 0) and (delta >= pooled) and (t2_9 >= t2_6 - 1e-6):
        verdict = (f"GO Stage 2 : 9-node F1@p99 +{delta:.4f} over 6-node "
                   f"(>= pooled std {pooled:.4f}) AND tail_R2 not worse")
    elif delta > 0:
        verdict = (f"WEAK +{delta:.4f} F1@p99 but within noise (pooled std "
                   f"{pooled:.4f}) -- not a robust GO")
    else:
        verdict = (f"NO-GO : 9-node F1@p99 {delta:+.4f} vs 6-node -- humid chain "
                   f"does not improve Stage-1 tail skill; save Stage-2 compute")

print("\n  VERDICT :", verdict)
_GATE_VERDICT = {
    "f1_9node_mean": f1_9, "f1_9node_std": s1_9, "n9": n9,
    "f1_6node_mean": f1_6, "f1_6node_std": s1_6, "n6": n6,
    "tail_r2_9node": t2_9, "tail_r2_6node": t2_6, "verdict": verdict,
}
if _DIR9.exists():
    (_DIR9 / "stage1_gate_verdict.json").write_text(
        _json.dumps(_GATE_VERDICT, indent=2, default=str))
    print(f"  saved -> {_DIR9 / 'stage1_gate_verdict.json'}")

In [ ]:
# >>> Cell 7 : Per-seed Q_phys re-eval from disk (idempotent)
#
# Re-reads each seed's A_dag from {seed_dir}/epoch_last.pth or, if absent, from
# results.json. Recomputes Q_phys binary/adaptive/continuous/phys_mag_gained
# /skel_f1 with the current helper code. This makes the eval reproducible from
# disk artifacts only (no live stack required) and decoupled from cell 6's
# inline computation.
#
# Output : {seed_dir}/q_phys_metrics.json (overwrites)
print("\n" + "=" * 78)
print("[Cell 7] Per-seed Q_phys metrics from disk")
print("=" * 78)

for _seed in SEEDS:
    seed_dir = ORACLE_FULL_DIR / f"seed_{_seed}"
    results_path = seed_dir / "results.json"
    if not results_path.exists():
        print(f"  [SEED {_seed}] results.json missing -> skip")
        continue

    res = json.loads(results_path.read_text())
    A_dag_final = np.asarray(res["A_dag_final"])
    A_dag_initial = np.asarray(res["A_dag_initial"])

    qb, _, _, n_extra_bin = compute_q_phys_binary(A_dag_final, G_phys_np)
    qa, _, _, _thr_a = compute_q_phys_adaptive(A_dag_final, G_phys_np)
    qc, collapsed = compute_q_phys_continuous(A_dag_final, G_phys_np)
    phys_gained = compute_phys_mag_gained(A_dag_final, A_dag_initial, G_phys_np)
    skel, _thr_s = compute_skeleton_f1(A_dag_final, G_phys_np)

    metrics = {
        "seed": int(_seed),
        "q_phys_binary":     qb,
        "q_phys_adaptive":   qa,
        "q_phys_continuous": qc,
        "q_phys_collapsed":  bool(collapsed),
        "phys_mag_gained":   phys_gained,
        "skeleton_f1":       skel,
        "n_extra_edges":     int(n_extra_bin),
        "A_dag_final_norm_frobenius": float(np.linalg.norm(A_dag_final, ord="fro")),
        "A_dag_initial_norm_frobenius": float(np.linalg.norm(A_dag_initial, ord="fro")),
    }
    metrics = stamp_option_c_json(
        metrics, seed=_seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
        pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
        pre_registration_commit=PRE_REGISTRATION_COMMIT,
        is_aggregate=False,
    )
    out = seed_dir / "q_phys_metrics.json"
    out.write_text(json.dumps(metrics, indent=2, default=str))
    print(f"  [SEED {_seed}] Q_phys_cont={qc:.4f} "
          f"(bin={qb:.4f}, adap={qa:.4f}, phys_mag_gained={phys_gained:.4f}, "
          f"skel_f1={skel:.4f}, n_extra={n_extra_bin}) -> {out.name}")

print("[Cell 7] done.")


In [ ]:
# >>> Cell 8 : Per-seed full eval pipeline (FINAL_VAL + BS42 + BS43 + 3x BS45/BS44)
#
# Council #5 fix : wrap the entire eval flow in `run_full_eval_for_seed(...)`
# so the globals (_pred_full, _pred_std, _sample_once, _ckpt_dir, etc.) used
# by the noncausal eval cells live inside a function scope and are GC'd
# between seeds. Prevents state leakage / closure capture across seeds.
#
# Each seed produces :
#   {seed_dir}/final_validation_metrics.json   (BS30 mirror)
#   {seed_dir}/domain_metrics.json             (BS42 mirror)
#   {seed_dir}/eval_samples.npz                (BS43 mirror)
#   {seed_dir}/aligned_metrics_<GCM>_causal.json  x 3  (BS45+BS44 mirror)
#
# The function loads the seed's epoch_last.pth weights into a fresh stack
# so it can also be re-run AFTER cell 6 completes for all seeds. Skip-test
# at top short-circuits already-complete seeds.

from st_cdgm.evaluation import compute_f1_extremes, compute_spectrum_distance
from st_cdgm.evaluation.two_stage_inference import build_two_stage_inputs
from st_cdgm.training.stage1_paths import resolve_run_variant
from st_cdgm.data.pipeline import NetCDFDataPipeline as _NCDP

# Note : ACCESS-CM2 train files live on SSD copy (BS32) ; test/ OOD files stay
# on Drive (not in BS32 copy list). _gcm_paths picks the right root per-file.
def _gcm_paths(rel_lr, rel_hr, in_dist):
    """Return (lr, hr, in_dist) with file existence check across SSD + Drive."""
    for _root in (DATA_ROOT, globals().get("DATA_ROOT_DRIVE"), DATA_ROOT_LOCAL):
        if _root is None:
            continue
        _lr = Path(_root) / rel_lr
        _hr = Path(_root) / rel_hr
        if _lr.exists() and _hr.exists():
            return (str(_lr), str(_hr), in_dist)
    # Fallback to DATA_ROOT_DRIVE even if missing (will raise downstream with clear path)
    _root = globals().get("DATA_ROOT_DRIVE", DATA_ROOT)
    return (str(Path(_root) / rel_lr), str(Path(_root) / rel_hr), in_dist)

GCM_REGISTRY = {
    "ACCESS-CM2": _gcm_paths(
        "train/predictor_ACCESS-CM2_hist.nc",
        "train/pr_ACCESS-CM2_hist.nc",
        True,
    ),
    "EC-Earth3": _gcm_paths(
        "test/EC-Earth3_histupdated_compressed.nc",
        "test/EC-Earth3_historical_precip_compressed.nc",
        False,
    ),
    "NorESM2-MM": _gcm_paths(
        "test/NorESM2-MM_histupdated_compressed.nc",
        "test/NorESM2-MM_historical_precip_compressed.nc",
        False,
    ),
}
for _g, _t in GCM_REGISTRY.items():
    print(f"  [GCM] {_g:12s} : LR={_t[0]}")
    print(f"  [GCM] {_g:12s} : HR={_t[1]}  in_dist={_t[2]}")


def _seed_eval_complete(seed: int) -> bool:
    """All 5 eval files present : final + domain + eval_samples + 3 GCM aligned."""
    d = ORACLE_FULL_DIR / f"seed_{seed}"
    must_exist = [
        d / "final_validation_metrics.json",
        d / "domain_metrics.json",
        d / "eval_samples.npz",
    ] + [d / f"aligned_metrics_{g}_causal.json" for g in GCMS]
    return all(p.exists() for p in must_exist)


def _pearson_torch(a, b, eps=1e-8):
    a_c = a - a.mean(); b_c = b - b.mean()
    num = (a_c * b_c).sum()
    den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
    return float((num / den).item())


def run_full_eval_for_seed(seed: int):
    """One-shot eval pipeline. Reloads checkpoint into fresh stack to isolate state."""
    seed_dir = ORACLE_FULL_DIR / f"seed_{seed}"
    if _seed_eval_complete(seed):
        print(f"  [SEED {seed}] eval already complete -- skip")
        return
    ck_path = seed_dir / "epoch_last.pth"
    if not ck_path.exists():
        print(f"  [SEED {seed}] no epoch_last.pth -- training first? skip")
        return

    print(f"\n  [SEED {seed}] running full eval pipeline...")
    stack = build_fresh_stack(seed)

    # Council AI Eng B1 FIX : materialize lazy SAGEConv params BEFORE state_dict load.
    # IntelligibleVariableEncoder uses in_channels=-1 lazy params. Without one
    # forward pass first, _persist_load_state_dict's shape check raises on
    # UninitializedParameter -> the matching loop silently drops every key ->
    # the encoder runs with FRESH (random) weights at eval. Every JSON would
    # be wrong. Force a no-grad forward to trigger PyG's lazy materialization.
    print(f"    [SAGEConv pre-warm] forcing lazy param materialization...")
    stack["encoder"].train(); stack["rcn_cell"].train(); stack["regression_head"].train()
    with torch.no_grad():
        for _conv in iterate_batches(train_dataloader, builder, DEVICE):
            for _b in _conv:
                _H = stack["encoder"].init_state(_b["hetero"]).to(DEVICE)
                _lr = _b["lr"].to(DEVICE)
                _drv = [_lr[t] for t in range(_lr.shape[0])]
                _seq = stack["rcn_runner"].run(_H, _drv, reconstruction_sources=None)
                _ = stack["regression_head"](_seq.states[-1])
                break
            break
    print(f"    [SAGEConv pre-warm] done -- params materialized")

    ck = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    _persist_load_state_dict(stack["encoder"],         ck.get("encoder_state_dict"))
    _persist_load_state_dict(stack["rcn_cell"],        ck.get("rcn_cell_state_dict"))
    _persist_load_state_dict(stack["regression_head"], ck.get("regression_head_state_dict"))
    _persist_load_state_dict(stack["spatial_projector"], ck.get("spatial_projector_state_dict"))
    if stack["hr_ident_head"] is not None:
        _persist_load_state_dict(stack["hr_ident_head"], ck.get("hr_ident_head_state_dict"))
    # Prefer EMA for inference if present
    diffusion = stack["diffusion"]
    ema_sd = ck.get("diffusion_ema_state_dict")
    if ema_sd is not None and not bool(globals().get("BS41_FORCE_LIVE_INFERENCE", False)):
        print(f"    EMA detected -- loading EMA weights into diffusion")
        _persist_load_state_dict(diffusion, ema_sd)
    else:
        _persist_load_state_dict(diffusion, ck.get("diffusion_state_dict"))

    # Restore sigma_data if saved
    ts = ck.get("two_stage") or {}
    if ts.get("sigma_data") is not None:
        from st_cdgm.models.edm_preconditioner import EDMConfig as _EDMConfig
        diffusion.edm_config = _EDMConfig(
            sigma_data=float(ts["sigma_data"]),
            sigma_min=float(ts.get("sigma_min", max(1e-4, float(ts["sigma_data"]) * 0.02))),
            sigma_max=float(CONFIG.diffusion.edm.sigma_max),
            rho=float(CONFIG.diffusion.edm.rho),
            P_mean=float(CONFIG.diffusion.edm.P_mean),
            P_std=float(CONFIG.diffusion.edm.P_std),
        )

    encoder = stack["encoder"]; rcn_cell = stack["rcn_cell"]
    rcn_runner = stack["rcn_runner"]; regression_head = stack["regression_head"]
    encoder.eval(); rcn_runner.cell.eval(); regression_head.eval(); diffusion.eval()

    # ------- helpers (build_two_stage_inputs + sampler closure) ---------------
    _diff_core = getattr(diffusion, "_orig_mod", diffusion)
    _diff_core = getattr(_diff_core, "module", _diff_core)
    _causal_concat = bool(getattr(_diff_core, "causal_concat", False))
    RUN_VARIANT_EVAL = resolve_run_variant(CONFIG)

    def _build_inputs(_batch):
        return build_two_stage_inputs(
            _batch, variant=RUN_VARIANT_EVAL,
            regression_head=regression_head,
            encoder=encoder if RUN_VARIANT_EVAL == "causal" else None,
            rcn_runner=rcn_runner if RUN_VARIANT_EVAL == "causal" else None,
            builder=builder, device=DEVICE,
        )

    _EVAL_SCHEDULER = str(CONFIG.diffusion.get(
        "scheduler_type", "edm_karras" if _causal_concat else "dpm_solver++"))
    # J29 audit fix : CFG (cfg_scale>1.0) is NOT implemented in _sample_edm_karras.
    # corrdiff_normal.yaml still carries cfg_scale=1.5 from the V4 0.815 run — force 1.0.
    _CONFIG_CFG = float(CONFIG.diffusion.get("cfg_scale", 1.0))
    if _EVAL_SCHEDULER == "edm_karras":
        if _CONFIG_CFG > 1.0 + 1e-9:
            print(f"[J29] CONFIG cfg_scale={_CONFIG_CFG} but scheduler=edm_karras "
                  f"(no CFG) -> forcing cfg_scale=1.0")
        _EVAL_CFG_SCALE = 1.0   # always 1.0 for edm_karras (never pass 1.5)
    else:
        _EVAL_CFG_SCALE = _CONFIG_CFG
    print(f"[sampler] scheduler={_EVAL_SCHEDULER}, cfg_scale={_EVAL_CFG_SCALE}")

    def _sample_once(_cond, _mu_HR, _baseline_log):
        kw = dict(
            num_steps=int(CONFIG.diffusion.get(
                "eval_num_steps", 18 if _causal_concat else 30)),
            scheduler_type=_EVAL_SCHEDULER,
            cfg_scale=_EVAL_CFG_SCALE,
            apply_constraints=False,
        )
        if _causal_concat:
            kw["mu_HR"] = _mu_HR
            kw["baseline_log"] = _baseline_log
        return _diff_core.sample(conditioning=_cond, **kw).residual

    # ------- 1. FINAL_VAL (BS30 mirror) --------------------------------------
    # Council SRE I3 : per-file skip avoids ~50min wasted re-sampling on partial re-entry.
    _fv_path = seed_dir / "final_validation_metrics.json"
    _dm_path = seed_dir / "domain_metrics.json"
    _es_path = seed_dir / "eval_samples.npz"
    _need_fv = not _fv_path.exists()
    _need_dm = not _dm_path.exists()
    _need_es = not _es_path.exists()
    _need_sampling = _need_fv or _need_dm or _need_es
    if not _need_sampling:
        print(f"    [BS30] FINAL_VAL + DOMAIN + EVAL_SAMPLES all exist -- skip sampling")
        # Still need _build_inputs + _sample_once for BS44 below; they are closures
        # already defined above so this is OK.
        _pred_full = None  # marker so BS43 export skips
    if _need_sampling:
        print(f"    [BS30] FINAL_VAL sampling...")
    N_TEST_BATCHES = 16
    K_SAMPLES = int(globals().get("K_SAMPLES_OVERRIDE", 64))
    N_INTERVENTION = 4

    # Council SRE I3 : skip the expensive K=64 sampling loop entirely if all
    # 3 outputs (FINAL_VAL + DOMAIN + EVAL_SAMPLES) already exist on disk.
    if not _need_sampling:
        # Set dummy variables so downstream blocks (which still run for BS44
        # GCM eval) don't NameError. BS44 closures _build_inputs + _sample_once
        # are defined above and remain valid.
        _pred_mean = _pred_std = _targets = _pred_full = None
        _mu_concat = None; _valid = None
        _rmse = _mae = _spread = _corr_global = _corr_per_sample = float("nan")
        _f1 = {}; _rapsd_d = None; _intervention = []; _dag_avg = None
        _mu_verdict = "skipped_existing"; _shortcut = {"verdict": "skipped_existing"}
        _metrics_scope = "skipped_existing"; _eval_time = 0.0
        _corr_per_sample_list = []

    _all_means, _all_stds, _all_targets, _all_mu_HR = [], [], [], []
    _intervention = []
    _t0 = time.time(); _count = 0
    if not _need_sampling:
        _count = N_TEST_BATCHES  # bypass the loop
    with torch.no_grad():
        for converted in iterate_batches(val_dataloader, builder, DEVICE):
            for _b in converted:
                if _count >= N_TEST_BATCHES:
                    break
                _cond, _mu_HR, _blog, _tgt = _build_inputs(_b)
                _samp = torch.stack(
                    [_sample_once(_cond, _mu_HR, _blog) for _ in range(K_SAMPLES)],
                    dim=0,
                )
                _all_means.append(_samp.mean(dim=0))
                _all_stds.append(_samp.std(dim=0))
                _all_targets.append(_tgt)
                if _causal_concat and _mu_HR is not None:
                    _all_mu_HR.append(_mu_HR.detach())
                if _count < N_INTERVENTION and _causal_concat and _mu_HR is not None:
                    _mu_zero = torch.zeros_like(_mu_HR)
                    _s_real = _sample_once(_cond, _mu_HR, _blog)
                    _s_zero = _sample_once(_cond, _mu_zero, _blog)
                    _d = (_s_real - _s_zero).abs().mean().item()
                    _sig = _s_real.abs().mean().item()
                    _intervention.append(_d / max(_sig, 1e-8))
                _count += 1
            if _count >= N_TEST_BATCHES:
                break
    _eval_time = time.time() - _t0

    if not _need_sampling:
        print(f"    [BS30] metrics on disk -- skip recompute (torch.cat)")
        if _fv_path.exists():
            try:
                _fv_loaded = json.loads(_fv_path.read_text())
                print(f"    [BS30] cached RMSE={_fv_loaded.get('rmse')}, "
                      f"Pearson={_fv_loaded.get('pearson_corr', {}).get('per_sample_avg')}")
            except Exception:
                pass
    else:
        _pred_mean = torch.cat(_all_means, dim=0).cpu()
        _pred_std  = torch.cat(_all_stds, dim=0).cpu()
        _targets   = torch.cat(_all_targets, dim=0).cpu()
    _valid = torch.isfinite(_targets)
    _mu_concat = None
    if _all_mu_HR:
        _mu_concat = torch.cat(_all_mu_HR, dim=0).cpu()
        if _mu_concat.shape == _pred_mean.shape:
            _pred_full = _pred_mean + _mu_concat
            _metrics_scope = "full_prediction (mu_HR + delta)"
        else:
            _pred_full = _pred_mean
            _metrics_scope = "delta_only (shape mismatch)"
    else:
        _pred_full = _pred_mean
        _metrics_scope = "delta_only (no mu_HR)"

    _pred_clean = torch.where(_valid, _pred_full, torch.zeros_like(_pred_full))
    _targ_clean = torch.where(_valid, _targets, torch.zeros_like(_targets))
    _rmse = float(((_pred_clean - _targ_clean) ** 2)[_valid].mean().sqrt().item()) if _valid.any() else float("nan")
    _mae  = float((_pred_clean - _targ_clean).abs()[_valid].mean().item()) if _valid.any() else float("nan")
    _spread = float(_pred_std[_valid].mean().item()) if _valid.any() else float("nan")

    _corr_global = float("nan"); _corr_per_sample_list = []
    try:
        _p_flat = _pred_full[_valid]; _t_flat = _targets[_valid]
        if _p_flat.numel() > 1:
            _corr_global = _pearson_torch(_p_flat, _t_flat)
        for _i in range(_pred_full.shape[0]):
            _vi = _valid[_i]
            if _vi.sum() < 2:
                continue
            _c = _pearson_torch(_pred_full[_i][_vi], _targets[_i][_vi])
            if _c == _c:
                _corr_per_sample_list.append(_c)
    except Exception as e:
        print(f"    Pearson failed: {e}")
    _corr_per_sample = float(np.mean(_corr_per_sample_list)) if _corr_per_sample_list else float("nan")

    _f1 = {}
    try:
        _f1 = compute_f1_extremes(_pred_clean, _targ_clean, threshold_percentiles=[95.0, 99.0])
    except Exception as e:
        print(f"    F1 failed: {e}")
    _rapsd_d = None
    try:
        _rapsd_d = float(compute_spectrum_distance(_pred_clean[0], _targ_clean[0]))
    except Exception as e:
        print(f"    RAPSD failed: {e}")

    _dag_avg = float(np.mean(_intervention)) if _intervention else None
    if _dag_avg is None:
        _mu_verdict = "N/A"
    elif _dag_avg < 0.001:
        _mu_verdict = "MU_HR_IGNORED"
    elif _dag_avg < 0.01:
        _mu_verdict = "WEAK"
    else:
        _mu_verdict = "MU_HR_CONDITIONS"

    _shortcut = {"verdict": "N/A"}
    if _mu_concat is not None and _mu_concat.shape == _pred_mean.shape:
        _v_mu = _valid & torch.isfinite(_mu_concat)
        _out = _pred_mean[_v_mu]; _mu = _mu_concat[_v_mu]; _tg = _targets[_v_mu]
        _shortcut = {
            "norm_output": float(_out.abs().mean().item()),
            "norm_mu_HR":  float(_mu.abs().mean().item()),
            "norm_target": float(_tg.abs().mean().item()),
            "norm_output_minus_mu_HR": float((_out - _mu).abs().mean().item()),
            "norm_target_minus_mu_HR": float((_tg - _mu).abs().mean().item()),
        }
        _r = _shortcut["norm_output_minus_mu_HR"] / max(_shortcut["norm_output"], 1e-12)
        _shortcut["shortcut_ratio"] = float(_r)
        _shortcut["verdict"] = ("SHORTCUT_CONFIRMED" if _r < 0.10
                                 else "AMBIGUOUS" if _r < 0.30
                                 else "REFINEMENT_OK")

    final_metrics = {
        "checkpoint": str(ck_path),
        "epoch": ck.get("epoch"),
        "epochs_total": ck.get("epochs_total"),
        "causal_concat": _causal_concat,
        "n_test_batches": len(_all_targets),
        "k_samples": K_SAMPLES,
        "metrics_scope": _metrics_scope,
        "eval_time_s": _eval_time,
        "rmse": _rmse, "mae": _mae, "spread_mean": _spread,
        "f1_extremes": _f1,
        "pearson_corr": {
            "global": _corr_global,
            "per_sample_avg": _corr_per_sample,
            "per_sample_n": len(_corr_per_sample_list),
            "per_sample_list": _corr_per_sample_list[:64],
        },
        "rapsd_distance": _rapsd_d,
        "mu_HR_ablation": {
            "delta_signal_ratio_avg": _dag_avg,
            "per_batch": _intervention,
            "verdict": _mu_verdict,
        },
        "shortcut_diagnostic": _shortcut,
        "config_eval_num_steps": int(CONFIG.diffusion.get(
            "eval_num_steps", 18 if _causal_concat else 30)),
        "config_cfg_scale": float(_CONFIG_CFG),
        "eval_cfg_scale": float(_EVAL_CFG_SCALE),
        "config_block_out_channels": list(CONFIG.diffusion.unet_kwargs.block_out_channels),
    }
    if _need_fv:
        final_metrics = stamp_option_c_json(
            final_metrics, seed=seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
            pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
            pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=False,
        )
        _fv_path.write_text(json.dumps(final_metrics, indent=2, default=str))
        print(f"    [BS30] RMSE={_rmse:.4f}, MAE={_mae:.4f}, "
              f"Pearson={_corr_per_sample:.4f}, mu_HR verdict={_mu_verdict}")
    else:
        print(f"    [BS30] final_validation_metrics.json exists -- skip write")

    # ------- 2. BS42 DOMAIN METRICS ------------------------------------------
    print(f"    [BS42] DOMAIN metrics...")
    import math as _m42
    _ssr = float(_spread) / float(_rmse) if (_rmse == _rmse and _rmse > 0) else float("nan")
    try:
        _mu_c = _pred_full[_valid].double()
        _sd_c = _pred_std[_valid].double().clamp_min(1e-6)
        _y_c = _targets[_valid].double()
        _w_c = (_y_c - _mu_c) / _sd_c
        _Phi = 0.5 * (1.0 + torch.erf(_w_c / _m42.sqrt(2.0)))
        _phi = torch.exp(-0.5 * _w_c * _w_c) / _m42.sqrt(2.0 * _m42.pi)
        _crps_pix = _sd_c * (_w_c * (2.0 * _Phi - 1.0) + 2.0 * _phi - 1.0 / _m42.sqrt(_m42.pi))
        _crps = float(_crps_pix.mean().item())
    except Exception:
        _crps = float("nan")
    try:
        _p_h = _pred_full[_valid].double().flatten()
        _t_h = _targets[_valid].double().flatten()
        _lo = float(torch.minimum(_p_h.min(), _t_h.min()).item())
        _hi = float(torch.maximum(_p_h.max(), _t_h.max()).item())
        if _hi > _lo:
            _hp = torch.histc(_p_h.float(), bins=100, min=_lo, max=_hi)
            _ht = torch.histc(_t_h.float(), bins=100, min=_lo, max=_hi)
            _hp = _hp / _hp.sum().clamp_min(1.0); _ht = _ht / _ht.sum().clamp_min(1.0)
            _lhd = float((_hp - _ht).abs().sum().item())
        else:
            _lhd = float("nan")
    except Exception:
        _lhd = float("nan")
    domain_metrics = {
        "spread_skill_ratio": _ssr,
        "crps_gaussian": _crps,
        "intensity_hist_distance_L1": _lhd,
        "rapsd_distance": float(_rapsd_d) if _rapsd_d is not None else None,
        "rmse_secondary": float(_rmse),
        "mae_secondary": float(_mae),
        "spread_mean": float(_spread),
        "pearson_global_secondary": float(_corr_global) if _corr_global == _corr_global else None,
    }
    if _need_dm:
        domain_metrics = stamp_option_c_json(
            domain_metrics, seed=seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
            pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
            pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=False,
        )
        _dm_path.write_text(json.dumps(domain_metrics, indent=2, default=str))
        print(f"    [BS42] SSR={_ssr:.4f}, CRPS={_crps:.6f}, LHD={_lhd:.4f}")
    else:
        print(f"    [BS42] domain_metrics.json exists -- skip write")

    # ------- 3. BS43 EVAL_SAMPLES export -------------------------------------
    print(f"    [BS43] eval_samples.npz export...")
    _N43 = min(8, int(_targets.shape[0]))
    def _np_sub(t):
        return t[:_N43].detach().float().cpu().numpy() if hasattr(t, "detach") else np.asarray(t)[:_N43]
    payload43 = dict(
        target=_np_sub(_targets), pred_full=_np_sub(_pred_full),
        pred_std=_np_sub(_pred_std), valid_mask=_np_sub(_valid).astype("float32"),
        run_variant=np.array("causal"),
    )
    if _mu_concat is not None:
        payload43["mu_HR"] = _np_sub(_mu_concat)
    if _need_es:
        np.savez_compressed(_es_path, **payload43)
        print(f"    [BS43] eval_samples.npz : N={_N43}")
    else:
        print(f"    [BS43] eval_samples.npz exists -- skip write")

    # ------- 4. BS45 + BS44 : loop over 3 GCMs -------------------------------
    from st_cdgm.evaluation.aligned_eval import run_aligned_eval
    for _gcm_tag in GCMS:
        out_aligned = seed_dir / f"aligned_metrics_{_gcm_tag}_causal.json"
        if out_aligned.exists():
            print(f"    [BS44/{_gcm_tag}] already exists -- skip")
            continue
        print(f"    [BS45/{_gcm_tag}] building dataloader...")
        _lr_path, _hr_path, _in_dist = GCM_REGISTRY[_gcm_tag]
        _pipe = _NCDP(
            lr_path=_lr_path, hr_path=_hr_path,
            static_path=STATIC_PATH, seq_len=SEQ_LEN,
            baseline_strategy=BASELINE_STRATEGY, baseline_factor=BASELINE_FACTOR,
            normalize=NORMALIZE, nan_fill_strategy=NAN_FILL_STRATEGY,
            precipitation_delta=PRECIPITATION_DELTA,
            lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
            static_variables=STATIC_VARIABLES,
            means_path=MEAN_PATH if (MEAN_PATH and os.path.exists(MEAN_PATH)) else None,
            stds_path=STD_PATH if (STD_PATH and os.path.exists(STD_PATH)) else None,
        )
        _ds_g = _pipe.build_sequence_dataset(
            seq_len=SEQ_LEN, stride=1, drop_last=True, as_torch=True, training=False,
        )
        from torch.utils.data import DataLoader as _DLg, IterableDataset as _IDg
        _dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
        if not isinstance(_ds_g, _IDg):
            _dl_kw["shuffle"] = False
        ALIGNED_EVAL_LOADER = _DLg(_ds_g, **_dl_kw)

        print(f"    [BS44/{_gcm_tag}] running aligned eval (in_dist={_in_dist})...")
        _Kal = int(globals().get("ALIGNED_K_SAMPLES", min(K_SAMPLES, 16)))
        _pred_seq, _truth_seq, _time_seq = [], [], []
        # Council DS B1 + Math Prof B2 : compute CRPS Gaussian per-GCM inline.
        # We already have K=16 ensemble samples per batch; use mean + std for
        # closed-form CRPS in log1p space (matches BS42 formula).
        _crps_accum = []
        with torch.no_grad():
            for _conv in iterate_batches(ALIGNED_EVAL_LOADER, builder, DEVICE):
                for _b in _conv:
                    _cond, _muHR, _blog, _tgt = _build_inputs(_b)
                    _samples = torch.stack(
                        [_sample_once(_cond, _muHR, _blog) for _ in range(_Kal)],
                        dim=0,
                    )
                    _ens = _samples.mean(dim=0)
                    _ens_std = _samples.std(dim=0)
                    _full_log = (_blog if _blog is not None else 0.0) + \
                                (_muHR if _muHR is not None else 0.0) + _ens
                    _truth_log = (_blog if _blog is not None else 0.0) + \
                                 (_muHR if _muHR is not None else 0.0) + _tgt
                    # CRPS Gaussian (closed-form, log1p space) on residual scale
                    try:
                        import math as _mc
                        _mu_c = _ens.double()
                        _sd_c = _ens_std.double().clamp_min(1e-6)
                        _y_c = _tgt.double()
                        _v_msk = torch.isfinite(_y_c) & torch.isfinite(_mu_c)
                        if _v_msk.any():
                            _w = (_y_c[_v_msk] - _mu_c[_v_msk]) / _sd_c[_v_msk]
                            _Phi = 0.5 * (1.0 + torch.erf(_w / _mc.sqrt(2.0)))
                            _phi = torch.exp(-0.5 * _w * _w) / _mc.sqrt(2.0 * _mc.pi)
                            _crps_pix = _sd_c[_v_msk] * (_w * (2.0 * _Phi - 1.0) + 2.0 * _phi - 1.0 / _mc.sqrt(_mc.pi))
                            _crps_accum.append(float(_crps_pix.mean().item()))
                    except Exception as _e_crps:
                        pass
                    for _i in range(_full_log.shape[0]):
                        _pred_seq.append(_full_log[_i].squeeze().detach().float().cpu().numpy())
                        _truth_seq.append(_truth_log[_i].squeeze().detach().float().cpu().numpy())
                    _tt = _b.get("time", None)
                    if _tt is not None:
                        _arr = np.atleast_1d(np.asarray(_tt))
                        _time_seq.append(_arr.ravel()[-1])
        _crps_gaussian_gcm = float(np.mean(_crps_accum)) if _crps_accum else float("nan")
        print(f"    [BS44/{_gcm_tag}] CRPS Gaussian = {_crps_gaussian_gcm:.6f} "
              f"(over {len(_crps_accum)} batches)")
        _T = len(_pred_seq)
        if _T == 0:
            print(f"    [BS44/{_gcm_tag}] no predictions -- skip")
            continue
        if len(_time_seq) == _T:
            _times = np.asarray(_time_seq, dtype="datetime64[ns]")
        else:
            _times = np.arange(_T, dtype="datetime64[D]").astype("datetime64[ns]")
        _res44 = run_aligned_eval(
            pred_fields=_pred_seq, truth_fields=_truth_seq, times=_times,
            out_path=out_aligned, gcm=_gcm_tag, run_variant="causal",
            in_distribution=_in_dist, space="log1p", thresh=1.0, k_samples=_Kal,
        )
        print(f"    [BS44/{_gcm_tag}] {_T} days, PSD distance={_res44['psd_distance']:.5f} -> {out_aligned.name}")

        # Council DS B1 + Math Prof B2 : inject per-GCM CRPS into aligned JSON
        # so cell 10 H5 can read OOD CRPS (NOT in-dist domain CRPS).
        _aligned_j = json.loads(out_aligned.read_text())
        _aligned_j["crps_gaussian_log1p"] = _crps_gaussian_gcm
        _aligned_j["crps_n_batches_aggregated"] = len(_crps_accum)
        out_aligned.write_text(json.dumps(_aligned_j, indent=2, default=str))

        # K23 OOD caveat appended to JSON for OOD GCMs
        if not _in_dist:
            _aligned_payload = json.loads(out_aligned.read_text())
            _aligned_payload["ood_limitation_k23"] = (
                "OOD evaluation uses same predictor mean/std as in-distribution train "
                "(ACCESS-CM2). Cross-GCM distribution shift is NOT corrected. OOD claim "
                "is restricted to: robustness under shared predictor distribution assumption."
            )
            _aligned_payload = stamp_option_c_json(
                _aligned_payload, seed=seed, seeds_lineage=SEEDS, k9_dates=K9_DATES,
                pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
                pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=False,
            )
            out_aligned.write_text(json.dumps(_aligned_payload, indent=2, default=str))

    # Free memory for this seed
    del stack, diffusion, _all_means, _all_stds, _all_targets, _all_mu_HR
    del _pred_mean, _pred_std, _targets, _pred_full, _pred_clean, _targ_clean
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Run eval for every seed that has a checkpoint but not yet a full eval set
print("\n" + "=" * 78)
print("[Cell 8] Per-seed eval pipeline")
print("=" * 78)
for _seed in SEEDS:
    run_full_eval_for_seed(_seed)

print("\n[Cell 8] done.")


In [ ]:
# >>> Cell 9 : Cross-seed H1 verdict (PC5 + PC8 5-AND-gate)
#
# Reads {seed_dir}/results.json for every seed in SEEDS, extracts the primary
# endpoint (Q_phys_continuous), then runs compute_h1_verdict from
# option_c_helpers with the 5-condition AND-gate :
#   mean_cont >= 0.50 AND bca_lower > baseline AND student_t_lower > baseline
#   AND bca_lower > random_null AND cohens_d >= 0.8
# Plus PC6 sparse_recovery flag (max n_extra < 3) and PC5-bis collapse check.
#
# Math Prof PASS-WITH-NITS : baseline_source MUST be specified explicitly. For
# Option C we use "noncausal_a_dag_absent" (true noncausal model has no learned
# A_dag = baseline 0). This is the most defensible choice : Q_phys_cont > 0
# is already a meaningful signal vs the noncausal baseline.
print("\n" + "=" * 78)
print("[Cell 9] Cross-seed H1 verdict (PC5 + PC8)")
print("=" * 78)

q_cont_per_seed = []
n_extra_per_seed = []
collapsed_per_seed = []
seeds_loaded = []

for _seed in SEEDS:
    _p = ORACLE_FULL_DIR / f"seed_{_seed}" / "results.json"
    if not _p.exists():
        print(f"  [SEED {_seed}] results.json MISSING -- aborting H1 verdict")
        break
    _r = json.loads(_p.read_text())
    q_cont_per_seed.append(float(_r["q_phys_continuous_final"]))
    n_extra_per_seed.append(int(_r["n_extra_edges_final"]))
    collapsed_per_seed.append(bool(_r.get("q_phys_collapsed_final", False)))
    seeds_loaded.append(_seed)

if len(q_cont_per_seed) < 3:
    print(f"  Not enough seeds ({len(q_cont_per_seed)}) -- need 3 to compute H1 verdict")
    verdict = {
        "verdict": "INCOMPLETE",
        "verdict_message": f"Only {len(q_cont_per_seed)}/3 seeds available.",
        "n_seeds": len(q_cont_per_seed),
        "seeds_loaded": seeds_loaded,
    }
else:
    print(f"  q_phys_continuous per seed = {q_cont_per_seed}")
    print(f"  n_extra_edges per seed     = {n_extra_per_seed}")
    print(f"  collapsed per seed         = {collapsed_per_seed}")

    # baseline_source = "noncausal_a_dag_absent" : noncausal has no A_dag
    # so baseline_cont = 0. Q_phys_cont > 0 is already meaningful evidence.
    #
    # Council Math Prof B1 FIX : recompute random_null from the ACTUAL G_phys
    # shape instead of hard-coding 0.083 (which assumes 6x6 / 4-node skeleton).
    # The iid-Gaussian null Q_phys = 0.5 * (n_phys / n_off_diag) where
    # n_off_diag = NUM_VARS * (NUM_VARS - 1).
    _n_phys_edges = int((G_phys_np != 0).sum())
    _n_off_diag = int(NUM_VARS * (NUM_VARS - 1))
    _random_null_computed = 0.5 * _n_phys_edges / max(_n_off_diag, 1)
    print(f"\n  [random_null] G_phys has {_n_phys_edges} edges / {_n_off_diag} off-diag entries")
    print(f"  [random_null] computed iid-Gaussian null = {_random_null_computed:.4f}")
    if abs(_random_null_computed - 0.083) > 0.01:
        print(f"  [random_null] WARNING : computed null {_random_null_computed:.4f} differs "
              f"from pre-reg 0.083. Using computed value (Math Prof B1).")

    verdict = compute_h1_verdict(
        q_cont_per_seed=q_cont_per_seed,
        n_extra_per_seed=n_extra_per_seed,
        collapsed_per_seed=collapsed_per_seed,
        baseline_cont=0.0,
        baseline_source="noncausal_a_dag_absent",
        random_null=_random_null_computed,
        h1_threshold=0.50,
        reporting_floor=0.30,
    )
    verdict["random_null_computed_from_G_phys"] = _random_null_computed
    verdict["random_null_pre_reg_value"] = 0.083
    verdict["n_phys_edges"] = _n_phys_edges
    verdict["n_off_diag_entries"] = _n_off_diag

    # Math Prof I6 : add Hedges' g (small-sample bias correction of Cohen's d).
    # g = d * J where J = 1 - 3/(4*df - 1), df = n-1. For n=3, J ~ 0.571.
    _n = len(q_cont_per_seed)
    _df = _n - 1
    _hedges_J = 1.0 - 3.0 / max(4 * _df - 1, 1)
    verdict["hedges_g_vs_baseline"] = float(verdict["cohens_d_vs_baseline"] * _hedges_J)
    verdict["hedges_J_correction"] = _hedges_J
    verdict["cohens_d_note"] = (
        "Cohen's d uses sd_floor=0.05 (Q_phys_cont is bounded [0,1]). With baseline=0, "
        "d >= 10 when mean >= 0.50 -- d is a CONSERVATIVE LOWER BOUND. Hedges' g "
        f"corrects for n=3 small-sample bias (J={_hedges_J:.3f})."
    )
    verdict["seeds_loaded"] = seeds_loaded
    print(f"\n  VERDICT : {verdict['verdict']}")
    print(f"  {verdict['verdict_message']}")
    print(f"  mean Q_phys_cont = {verdict['mean_q_phys_continuous']:.4f} "
          f"(sd={verdict['sd_q_phys_continuous']:.4f})")
    print(f"  BCa CI 95%       = [{verdict['bca_ci_95'][0]:.4f}, "
          f"{verdict['bca_ci_95'][1]:.4f}]")
    print(f"  Student-t lower  = {verdict['student_t_lower_95']:.4f}")
    print(f"  Cohen's d        = {verdict['cohens_d_vs_baseline']:.2f}")
    print(f"  H1 strict pass   = {verdict['h1_strict_pass']}")
    print(f"  PC6 sparse rec.  = {verdict['sparse_recovery_pc6']} "
          f"(max n_extra={max(n_extra_per_seed)})")

verdict = stamp_option_c_json(
    verdict, seeds_lineage=SEEDS, k9_dates=K9_DATES,
    pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
    pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=True,
)
H1_VERDICT_PATH = ORACLE_FULL_DIR / "h1_verdict.json"
H1_VERDICT_PATH.write_text(json.dumps(verdict, indent=2, default=str))
print(f"\n  saved -> {H1_VERDICT_PATH}")


In [ ]:
# >>> Cell 10 : Cross-seed H2-H5 tests vs noncausal baseline (Holm k=4)
#
# H2 : MAE (lower is better)  -- final_validation_metrics.json
# H3 : CDD bias (closer to 0) -- aligned_metrics_ACCESS-CM2_causal.json
# H4 : Spread-skill ratio (closer to 1) -- domain_metrics.json
# H5 : OOD CRPS (lower is better) -- domain_metrics.json on OOD GCM ; we use
#      the aligned ACCESS-CM2 noncausal CRPS (in-dist proxy here -- in absence
#      of an OOD CRPS in noncausal artifacts, fall back to domain CRPS)
#
# Each test runs one_sample_t_vs_noncausal_constant : Oracle 3 seeds vs the
# noncausal scalar baseline (treated as zero-variance constant). Direction
# is set per metric. Then holm_bonferroni_h2_h5 corrects across the family
# of k=4 (Math Prof + DS rename of Council #7, family excludes H1).

print("\n" + "=" * 78)
print("[Cell 10] H2-H5 vs noncausal (one-sample t + Holm-Bonferroni k=4)")
print("=" * 78)

# Load noncausal baseline JSONs (all that exist; some may be absent)
baseline = load_noncausal_baseline_metrics(CKPT_NONCAUSAL_DIR)
print(f"  noncausal baseline keys = {sorted(baseline.keys())}")

# Helper : robust dict-deep get
def _dig(d, *path, default=None):
    cur = d
    for k in path:
        if cur is None or not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


# Council DS B1 + Math Prof B2 FIX : H5 must use OOD CRPS (EC-Earth3 + NorESM2-MM
# averaged), NOT in-distribution ACCESS-CM2 CRPS from domain_metrics.json. Cell 8
# now writes `crps_gaussian_log1p` inside each per-GCM aligned JSON. We average
# the 2 OOD GCMs for the seed-level H5 value.
OOD_GCMS = ["EC-Earth3", "NorESM2-MM"]

# Collect Oracle per-seed metrics
oracle_per_seed = {h: [] for h in ["H2_MAE", "H3_CDD_bias", "H4_SSR", "H5_OOD_CRPS"]}
for _seed in SEEDS:
    seed_dir = ORACLE_FULL_DIR / f"seed_{_seed}"
    fv = seed_dir / "final_validation_metrics.json"
    dm = seed_dir / "domain_metrics.json"
    al = seed_dir / "aligned_metrics_ACCESS-CM2_causal.json"
    if not (fv.exists() and dm.exists() and al.exists()):
        print(f"  [SEED {_seed}] missing JSONs -- skip")
        continue
    fv_j = json.loads(fv.read_text())
    dm_j = json.loads(dm.read_text())
    al_j = json.loads(al.read_text())

    oracle_per_seed["H2_MAE"].append(float(fv_j.get("mae", float("nan"))))
    oracle_per_seed["H3_CDD_bias"].append(float(
        _dig(al_j, "indices", "cdd_bias", default=float("nan"))
    ))
    oracle_per_seed["H4_SSR"].append(float(dm_j.get("spread_skill_ratio", float("nan"))))
    # H5 : OOD CRPS averaged over EC-Earth3 + NorESM2-MM
    _ood_crps_seed = []
    for _gcm in OOD_GCMS:
        _al_ood = seed_dir / f"aligned_metrics_{_gcm}_causal.json"
        if _al_ood.exists():
            _v = float(json.loads(_al_ood.read_text()).get("crps_gaussian_log1p", float("nan")))
            if _v == _v:
                _ood_crps_seed.append(_v)
    oracle_per_seed["H5_OOD_CRPS"].append(
        float(np.mean(_ood_crps_seed)) if _ood_crps_seed else float("nan")
    )

print(f"\n  oracle_per_seed = {oracle_per_seed}")

# Noncausal scalar baseline
nc_h2 = float(_dig(baseline, "final_validation_metrics", "mae", default=float("nan")))
nc_h4 = float(_dig(baseline, "domain_metrics", "spread_skill_ratio", default=float("nan")))
nc_h3 = float(_dig(baseline, "aligned_metrics_per_gcm", "ACCESS-CM2",
                    "indices", "cdd_bias", default=float("nan")))
# H5 OOD CRPS noncausal : try to extract per-GCM CRPS from noncausal aligned JSONs
# if recorded (cell 8 writes it for Oracle, but noncausal ckpt_noncausal/ may
# not have it). Average over EC-Earth3 + NorESM2-MM if both exist; else NaN.
_nc_h5_ood = []
for _gcm in OOD_GCMS:
    _v = float(_dig(baseline, "aligned_metrics_per_gcm", _gcm,
                    "crps_gaussian_log1p", default=float("nan")))
    if _v == _v:
        _nc_h5_ood.append(_v)
nc_h5 = float(np.mean(_nc_h5_ood)) if _nc_h5_ood else float("nan")
if not _nc_h5_ood:
    print(f"  [H5] noncausal has NO per-GCM OOD CRPS in aligned JSONs.")
    print(f"       H5 will be SKIPPED. Holm-Bonferroni stays at k=4 (over-corrects, "
          f"per Math Prof B2). To recompute noncausal OOD CRPS, re-run noncausal "
          f"eval with the patched cell-8 inline CRPS block.")
print(f"  noncausal H2 MAE={nc_h2}, H3 CDD bias={nc_h3}, "
      f"H4 SSR={nc_h4}, H5 OOD CRPS={nc_h5}")

# Run one-sample t-test per H, direction-aware
h_tests = {}
def _safe_test(label, seed_vals, nc_val, direction, delta_threshold=None):
    if any(v != v for v in seed_vals) or nc_val != nc_val or len(seed_vals) < 2:
        return {"label": label, "test_type": "skipped_missing_data",
                "p_value_one_sided": float("nan"),
                "oracle_per_seed": list(seed_vals), "noncausal_value": nc_val,
                "direction": direction}
    return one_sample_t_vs_noncausal_constant(
        oracle_per_seed=seed_vals, noncausal_value=nc_val,
        label=label, delta_threshold=delta_threshold, direction=direction,
    )

h_tests["H2"] = _safe_test("H2_MAE", oracle_per_seed["H2_MAE"], nc_h2,
                            direction="lower_is_better")
# H3 CDD bias : closer to 0 -> we test |bias_oracle| < |bias_noncausal| via
# treating |bias| as the metric (lower is better).
oracle_h3_abs = [abs(v) for v in oracle_per_seed["H3_CDD_bias"]]
h_tests["H3"] = _safe_test("H3_CDD_abs_bias", oracle_h3_abs, abs(nc_h3) if nc_h3 == nc_h3 else nc_h3,
                            direction="lower_is_better")
# H4 SSR : closer to 1.0 -> |1 - SSR| lower is better
oracle_h4_dev = [abs(1.0 - v) for v in oracle_per_seed["H4_SSR"]]
h_tests["H4"] = _safe_test("H4_SSR_dev_from_1", oracle_h4_dev,
                            abs(1.0 - nc_h4) if nc_h4 == nc_h4 else nc_h4,
                            direction="lower_is_better")
h_tests["H5"] = _safe_test("H5_OOD_CRPS_avg", oracle_per_seed["H5_OOD_CRPS"], nc_h5,
                            direction="lower_is_better")

print(f"\n  Per-test p-values (raw, one-sided) :")
for k, v in h_tests.items():
    print(f"    {k} ({v['label']:25s}) : p={v.get('p_value_one_sided', float('nan')):.4f}, "
          f"oracle_mean={v.get('oracle_mean', float('nan')):.4f}, "
          f"noncausal={v.get('noncausal_value', float('nan')):.4f}")

# Holm-Bonferroni correction (k=4 -- PC14 #5)
p_raw = {k: v["p_value_one_sided"] for k, v in h_tests.items()
         if v["p_value_one_sided"] == v["p_value_one_sided"]}  # filter NaN
holm = holm_bonferroni_h2_h5(p_raw, alpha=0.05, family_size=4) if p_raw else {}

print(f"\n  Holm-Bonferroni (family size = 4) :")
for k, v in holm.items():
    print(f"    {k} : p_adj={v['p_adjusted']:.4f}, reject@0.05={v['reject_at_alpha']}")

# Council Math Prof I3 : document metric transforms used for H3 / H4
# Council Math Prof I5 : noncausal-as-zero-variance constant caveat
# Council DS B1 : H5 endpoint clarification
h2h5_payload = {
    "h_tests": h_tests,
    "holm_bonferroni_k4": holm,
    "noncausal_baseline_summary": {
        "H2_MAE_nc": nc_h2, "H3_CDD_bias_nc": nc_h3,
        "H4_SSR_nc": nc_h4, "H5_OOD_CRPS_nc": nc_h5,
    },
    "oracle_per_seed_summary": oracle_per_seed,
    "metric_transforms": {
        "H3_CDD_abs_bias": "abs(.) applied to oracle AND noncausal -- 'closer to 0' -> lower-is-better",
        "H4_SSR_dev_from_1": "abs(1 - .) applied to oracle AND noncausal -- 'closer to 1' -> lower-is-better",
    },
    "h2_h5_caveats": {
        "noncausal_as_zero_variance_constant": (
            "Noncausal baseline is a single trained model (epoch=200) treated as "
            "a known constant with zero variance. The test is one-sample Student-t "
            "(df=n-1=2), not paired. Conservative-for-Oracle when direction matches "
            "observed inequality; anti-conservative otherwise. Multi-seed noncausal "
            "would be required for a paired comparison and is deferred."
        ),
        "h5_endpoint": (
            "H5 = OOD CRPS averaged over EC-Earth3 + NorESM2-MM (NOT in-distribution "
            "ACCESS-CM2). Per Council DS B1 + Math Prof B2 fixes."
        ),
        "h5_skipped_if_noncausal_ood_crps_missing": (
            f"noncausal OOD CRPS present : {bool(_nc_h5_ood)}. If False, H5 p-value "
            f"is NaN and Holm-Bonferroni over-corrects (k=4 vs n_items=3)."
        ),
        "holm_family_size_k4_decision": (
            "Family size k=4 hard-coded per PC14 #5. When one test is NaN, k=4 stays "
            "(conservative over-correction). Alternative k=3 would require PC15 "
            "amendment -- not in scope for this run."
        ),
        "ood_limitation_k23": (
            "OOD evaluation uses same predictor mean/std as in-distribution train "
            "(ACCESS-CM2). Cross-GCM distribution shift is NOT corrected."
        ),
    },
}
h2h5_payload = stamp_option_c_json(
    h2h5_payload, seeds_lineage=SEEDS, k9_dates=K9_DATES,
    pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
    pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=True,
)
H2H5_PATH = ORACLE_FULL_DIR / "h2_h5_one_sample_tests.json"
H2H5_PATH.write_text(json.dumps(h2h5_payload, indent=2, default=str))
print(f"\n  saved -> {H2H5_PATH}")


In [ ]:
# >>> Cell 11 : Aggregate + export to results/oracle_*
#
# Consolidates h1_verdict + h2_h5_one_sample_tests + per-seed results.json into
# a single `oracle_full_aggregate.json`, applies stamp_option_c_json with
# `is_aggregate=True`, then copies the headline JSONs to `results/oracle_*`
# (NOT v4_* -- noncausal baseline lives in results/v4_metrics.json and must NOT
# be clobbered, per NONCAUSAL_EVAL_CONTRACT.md Run-variant switch checklist).

import shutil
print("\n" + "=" * 78)
print("[Cell 11] Aggregate + export")
print("=" * 78)

# Load per-seed result JSONs
per_seed_summaries = {}
for _seed in SEEDS:
    seed_dir = ORACLE_FULL_DIR / f"seed_{_seed}"
    res_p = seed_dir / "results.json"
    if not res_p.exists():
        per_seed_summaries[str(_seed)] = {"status": "MISSING_results_json"}
        continue
    res = json.loads(res_p.read_text())
    per_seed_summaries[str(_seed)] = {
        "q_phys_continuous": res["q_phys_continuous_final"],
        "q_phys_binary":     res["q_phys_binary_final"],
        "q_phys_adaptive":   res["q_phys_adaptive_final"],
        "q_phys_collapsed":  res["q_phys_collapsed_final"],
        "phys_mag_gained":   res["phys_mag_gained_final"],
        "skeleton_f1":       res["skeleton_f1_final"],
        "n_extra_edges":     res["n_extra_edges_final"],
        "sigma_data_post_s1": res["sigma_data_post_s1"],
        "ablation_report":   res["ablation_report"],
        "best_s1_val_mse":   res["best_s1_val_mse"],
    }

# Load cross-seed verdicts
h1 = json.loads(H1_VERDICT_PATH.read_text()) if H1_VERDICT_PATH.exists() else None
h2h5 = json.loads(H2H5_PATH.read_text()) if H2H5_PATH.exists() else None

aggregate = {
    "verdict_h1": h1,
    "h2_h5_one_sample_tests": h2h5,
    "per_seed_summaries": per_seed_summaries,
    "ood_limitation_k23": (
        "OOD evaluation uses same predictor mean/std as in-distribution train "
        "(ACCESS-CM2). Cross-GCM distribution shift is NOT corrected. OOD claim "
        "is restricted to: robustness under shared predictor distribution assumption."
    ),
    "pre_registration_protocol": "PRE_REGISTRATION.md PC1-PC14 (Option C primary, A1 preliminary)",
}
aggregate = stamp_option_c_json(
    aggregate, seeds_lineage=SEEDS, k9_dates=K9_DATES,
    pathcplus_hyperparams=PATHCPLUS_HYPERPARAM_OVERRIDES,
    pre_registration_commit=PRE_REGISTRATION_COMMIT, is_aggregate=True,
)
AGG_PATH = ORACLE_FULL_DIR / "oracle_full_aggregate.json"
AGG_PATH.write_text(json.dumps(aggregate, indent=2, default=str))
print(f"  saved aggregate -> {AGG_PATH}")

# Export to results/ (renamed targets so v4_metrics.json = noncausal stays intact)
_results_dir = Path.cwd() / "results"
_results_dir.mkdir(parents=True, exist_ok=True)

_export_map = {
    AGG_PATH: _results_dir / "oracle_full_aggregate.json",
    H1_VERDICT_PATH: _results_dir / "oracle_h1_verdict.json",
    H2H5_PATH: _results_dir / "oracle_h2_h5_one_sample_tests.json",
}
for _seed in SEEDS:
    sd = ORACLE_FULL_DIR / f"seed_{_seed}"
    _export_map[sd / "results.json"] = _results_dir / f"oracle_seed_{_seed}_results.json"
    _export_map[sd / "final_validation_metrics.json"] = (
        _results_dir / f"oracle_seed_{_seed}_final_validation_metrics.json"
    )
    _export_map[sd / "domain_metrics.json"] = (
        _results_dir / f"oracle_seed_{_seed}_domain_metrics.json"
    )
    for gcm in GCMS:
        _export_map[sd / f"aligned_metrics_{gcm}_causal.json"] = (
            _results_dir / f"oracle_seed_{_seed}_aligned_metrics_{gcm}_causal.json"
        )

print(f"\n  Copying {len(_export_map)} JSONs to results/oracle_* ...")
_copied = 0
for src, dst in _export_map.items():
    if src.exists():
        shutil.copy2(src, dst)
        _copied += 1
print(f"  copied {_copied}/{len(_export_map)} files")

# Final headline
print("\n" + "=" * 78)
print("OPTION C SUMMARY")
print("=" * 78)
if h1 is not None:
    print(f"  H1 verdict       : {h1.get('verdict', '?')}")
    print(f"  H1 message       : {h1.get('verdict_message', '')}")
    print(f"  mean Q_phys_cont : {h1.get('mean_q_phys_continuous', float('nan')):.4f}")
    print(f"  Cohen's d        : {h1.get('cohens_d_vs_baseline', float('nan')):.2f}")
    print(f"  BCa CI 95%       : {h1.get('bca_ci_95')}")
    print(f"  PC6 sparse rec.  : {h1.get('sparse_recovery_pc6')}")
if h2h5 is not None:
    print(f"\n  H2-H5 Holm-Bonferroni (k=4) :")
    for k, v in (h2h5.get("holm_bonferroni_k4") or {}).items():
        print(f"    {k} : p_adj={v.get('p_adjusted', float('nan')):.4f}, "
              f"reject@0.05={v.get('reject_at_alpha')}")
print(f"\n  Pre-registration commit : {PRE_REGISTRATION_COMMIT}")
print(f"  See : path_c_plus/PRE_REGISTRATION.md PC1-PC14")
print("=" * 78)
